# E17 — Fixed-Geometry Head Readout & Conditional Steering
## [해결책] 같은 head·split·steering을 고정하고, 내부 신호를 읽는 방식만 비교

**E15·E16과 별개인 E17입니다.** 원본 E14/E15/E16 폴더는 수정하지 않습니다.

> **실험 질문:** E14의 norm/attention-mass/shift 집계 대신, 같은 head의 벡터 방향을 반영하는 고정 기하학 규칙을 사용하면 K/C 상태 판별과 실제 QA 성능이 개선되는가?

**중요한 용어 정정:** 라벨별 평균차 방향·prototype도 넓은 의미에서 **supervised classifier/readout**입니다.
`predict_proba()`를 호출하지 않는다고 classifier-free가 되지는 않습니다.
본 실험에서 제거하는 것은 주 방법의 **별도 LR/MLP 최적화**이지, 라벨 의존성이나 분류 연산 자체가 아닙니다.
따라서 **optimizer-free geometric readout / supervised discovery**라고 보고합니다.

방향 투영은 여전히 소수 score를 계산합니다. 목표는 ‘scalar가 전혀 없는 방법’이 아니라
**크기만 보는 score 대신 방향을 반영하는 score를 사용하는 것**입니다.
PCA/centroid가 모든 정보를 보존하거나 LR보다 우수하다고 가정하지 않습니다.

## 0. 문제상황 → 원인 가설 → 해결책 → 평가

**문제상황:** E14의 별도 selected-head + LR 분석은 pooled 4-state Macro-F1 0.6306이었으나,
role-head scalar + median 진단은 0.3860, 실제 action과 K/C-derived target의 F1은 0.3126이었다.
기존 세 숫자는 head-set과 feature/판정 방식이 달라, 차이 전부를 ‘scalar화의 인과 효과’라고 할 수 없다.

**원인 가설:** (a) head별 값을 3개로 합치는 과정, (b) 벡터 방향을 버리는 과정,
(c) threshold/결합 규칙, (d) state-to-action 또는 actuator가 병목일 수 있다.

**해결책:** 각 outer fold의 E14 `HEAD_LABELFREE_POLICY_LOCK.json`에 들어 있는
memory/context/conflict **role-head 합집합**을 공통 입력 head 집합으로 고정한다.
같은 `HEAD_Q`, `HEAD_LAST`, `C_MASS`로 다음 readout을 비교한다.

| 비교군 | 입력/읽는 방식 | 추가 최적화 | 의미 |
|---|---|---|---|
| `e14_median4` | E14 memory/context score + PC median | 없음 | 기존 4-state 추가진단 재계산 |
| `e14_rule` | E14 memory/context/conflict + 원래 gate threshold | 없음 | 실제 3-action 정책 재실행 |
| `scalar_lr` | E14의 집계 scalar 3개 → LR | 있음 | 집계 scalar도 더 나은 결합 규칙으로 읽히는가 |
| `perhead_scalar_lr` | 같은 head별 norm/shift/mass → LR | 있음 | head별 scalar 집계에 따른 손실 진단 |
| `vector_lr` | 같은 head vector + C_MASS → LR | 있음 | 공통 feature의 supervised reference; upper bound 아님 |
| `direction` | fit K/C별 평균차 방향에 signed projection | LR/MLP 없음 | 방향을 반영하는 폐형식 readout |
| `prototype` | 표준화 벡터와 4개 상태 prototype의 cosine | LR/MLP 없음 | 상태별 대표 방향과의 유사도 |
| `pca_prototype` | fit-only PCA 32차원 → prototype | PCA만 | 저차원 geometry 보조실험 |
| random / shuffled controls | 무작위 방향 / fit 라벨을 섞은 prototype | LR/MLP 없음 | 방향·label association의 조건부 대조군 |

**효과 평가:** 4-state 판별 / action-target 일치도 / 실제 QA를 별도로 측정한다.
state F1 증가를 QA 개선으로 바꿔서 보고하지 않는다.
**E14의 0.6306을 E17 LR 값으로 재사용하지 않고 새로 계산한다.**

## 실행 방법

E14를 실행했던 **Drive 계정에 연결되는 별도의 Colab 노트북**에서 실행합니다.
기본 설정은 `RUN_GENERATION=True`, `QA_PROFILE="core"`이며 **처음부터 모두 실행**하면
캐시 분석 → 고정 readout 생성 → GPU 검사 → 실제 QA → 결과 보고서 순서로 진행합니다.
Hugging Face의 Llama 접근 권한과 Read token이 필요합니다.

먼저 상태 판별만 확인할 때는 `RUN_GENERATION=False`로 실행합니다.
그 뒤 같은 설정으로 True로 바꾸고 다시 실행하면 이미 잠근 readout을 재사용합니다.
**학습률/PCA 차원/head 선택 등 과학적 설정을 바꾸면 반드시 새 RUN_ID를 사용합니다.**

### 필수 E14 파일
```text
/content/drive/MyDrive/p8s/E14_L13_head_5fold_v1/
  L13_neutral_head_bank_all_rows.npz
  fold_0/ ... fold_4/
    split_manifest.csv
    HEAD_LABELFREE_POLICY_LOCK.json
    instruction_vectors.npz
```
경로가 다르면 `CFG["E14_ROOT"]`만 수정합니다. 파일이 없으면 명확히 중단합니다.
기존 노트북을 내려받는 것만으로 Colab Drive의 activation bank까지 복사되는 것은 아닙니다.

### 기본 QA 8개 조건
`noop`, `e14_rule`, `direction__budget`, `prototype__budget`, `prototype__state_map`,
`vector_lr__budget`, `scalar_lr__budget`, `prototype__budget_random_steer`.
각 1,224행, 총 **9,792개 답변 생성 실행**이 상한이며 batch별 중단 재개가 가능합니다.
CPU 분석은 기존 activation을 사용하지만 **QA는 실제 generation을 새로 실행**합니다.

In [ ]:
# 1. 실행 설정 — E14/E15 원본 폴더는 읽기만 합니다.
from pathlib import Path
import os

CFG = {
    "CODE_VERSION": "E17_geometry_v1.0.0",
    "RUN_ID": "E17_geometry_matched_heads_v1",
    "SEED": 42,
    "OUTER_FOLDS": [0, 1, 2, 3, 4],
    "E14_ROOT": "/content/drive/MyDrive/p8s/E14_L13_head_5fold_v1",
    "OUTPUT_PARENT": "/content/drive/MyDrive/p8s",
    "DATA_DIR": "/content/e17_input",
    "MODEL_REVISION_FILE": "/content/drive/MyDrive/p8s/E11_run01/model_revision.json",
    "MODEL_REVISION_OVERRIDE": "",  # E14의 정확한 SHA를 알면 지정. 비어 있으면 revision 파일 우선.
    "MODEL_ID": "meta-llama/Llama-3.1-8B-Instruct",
    "TRANSFORMERS_VERSION": "4.57.1",
    "SKLEARN_VERSION": "1.6.1",
    "READ_LAYER": 13,  # E14와 동일한 model.layers[13] (0-based)
    "WRITE_LAYERS": [14, 15, 16, 17, 18],
    "BATCH_SIZE": 4,
    "MAX_PROMPT_TOKENS": 4096,
    "MAX_NEW_TOKENS": 192,
    "ALPHA": 0.5,               # E14: scaled instruction vector에 곱하는 값
    "MAX_DELTA_RATIO": 0.05,    # E14: delta norm 상한. E15의 fixed-rho와 다름.
    "LR_C": 0.1,
    "PCA_DIM": 32,              # 사전 고정. test F1로 고르지 않음.
    "CONTROL_SEEDS": [101, 202, 303, 404, 505],
    "COVERAGE_GRID": [0.075, 0.15, 0.30, 0.50],  # 진단 곡선. test로 주 방법 선택 금지.
    "N_BOOT": 2000,
    "RUN_GENERATION": True,     # False: 기존 activation만으로 state/routing 분석. GPU 불필요.
    "QA_PROFILE": "core",      # core=8조건, extended=12조건. 아래 목록 참조.
    "STRICT_BANK_PARITY": True,
    "BANK_REL_ERROR_MAX": 0.02,
    "MASS_ABS_ERROR_MAX": 0.01,
    "RESUME": True,
}

QA_CORE = [
    "noop", "e14_rule", "direction__budget", "prototype__budget",
    "prototype__state_map", "vector_lr__budget", "scalar_lr__budget",
    "prototype__budget_random_steer",
]
QA_EXTENDED = QA_CORE + [
    "pca_prototype__budget", "always_retain", "always_accept",
    "random_direction_101__budget",
]
QA_METHODS = QA_CORE if CFG["QA_PROFILE"] == "core" else QA_EXTENDED
assert CFG["QA_PROFILE"] in {"core", "extended"}
assert CFG["MAX_NEW_TOKENS"] == 192
assert max([CFG["READ_LAYER"]]) < min(CFG["WRITE_LAYERS"])
assert sorted(CFG["OUTER_FOLDS"]) == list(range(5)), "완전한 OOF 분석은 5개 fold 전체가 필요합니다."
print("E17:", CFG["RUN_ID"], "| QA:", CFG["RUN_GENERATION"], "| conditions:", QA_METHODS)


In [ ]:
# 2. 환경 — 모델은 CPU 분석 완료 후 GPU 단계에서만 로드합니다.
import sys, subprocess, importlib.metadata
requirements = ["numpy", "pandas", "scipy", "scikit-learn==" + CFG["SKLEARN_VERSION"],
                "pyarrow", "gdown", "matplotlib", "threadpoolctl", "tabulate"]
if CFG["RUN_GENERATION"]:
    requirements += ["transformers==" + CFG["TRANSFORMERS_VERSION"], "accelerate", "safetensors"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *requirements])
# Colab에서 sklearn을 먼저 import한 뒤 버전을 바꾼 경우 세션을 재시작하세요.
import numpy as np
import pandas as pd
import sklearn
if sklearn.__version__ != CFG["SKLEARN_VERSION"]:
    raise RuntimeError("sklearn 버전이 메모리에 남았습니다. 세션 다시 시작 후 처음부터 실행하세요.")
from IPython.display import display, Markdown
from threadpoolctl import threadpool_limits
print("numpy", np.__version__, "pandas", pd.__version__, "sklearn", sklearn.__version__)


## 3. 데이터·split·재사용 규칙

기존 p8s 612질문 / 1,224행 및 `kside_corr` 라벨을 사용합니다.
각 질문의 C+/C− 두 행은 항상 같은 split에 둡니다.

| 단계 | 대략적 비중 | 사용 |
|---|---:|---|
| fit | 60% | 표준화, 방향/prototype, PCA, reference LR |
| utility_fit | 10% | E14에서 role-head를 선택했던 데이터; E17은 해당 fold lock 재사용 |
| policy_cal | 10% | 라벨 없이 margin/coverage threshold 설정 |
| test | 20% | 설정 완료 후 state/action/QA 평가 |

**금지:** test fold를 제외한다며 *다른 outer fold의 head 결과*를 모아 consensus를 만들지 않습니다.
다른 fold의 development에 현재 test 질문이 들어 있을 수 있기 때문입니다.
본 노트북은 **동일 outer fold의 E14 artifact만** 사용합니다.

이 데이터의 E14 결과를 이미 본 후 설계한 실험이므로,
‘완전히 새 blind test’가 아니라 **같은 벤치마크의 후속 통제 비교**로 보고합니다.

In [ ]:
# 3. 공통 함수: 데이터/라벨 검증, 재현 가능한 split, 원자적 저장
import json, hashlib, re, ast as pyast, string, unicodedata, time, gc, warnings, copy
from dataclasses import dataclass
from typing import Any
from scipy.special import logsumexp
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning

STATE_NAMES = ["K+C+", "K+C-", "K-C-", "K-C+"]
ACTION_NAMES = ["retain", "accept", "noop"]
SOURCE_FILES = {
    "behavior": ("1RXvW6o552ZNZn1PUMRaf8CEDMIUpqQOr", "p8s_behavior_v2.csv"),
    "klabels": ("1LMrQ3R4XWTcBbDn3kk2mR1iTITYw-KFF", "k_relabel_v2.parquet"),
}
SYSTEM_PROMPT = (
    "You are a question answering system. "
    "Answer the question based on the given context. "
    "Output the answer string itself and nothing else: no sentence, no explanation, "
    "no punctuation, no quotes. Keep it as short as possible (a name or a term)."
)

def to_jsonable(x: Any) -> Any:
    if isinstance(x, np.ndarray): return x.tolist()
    if isinstance(x, np.generic): return x.item()
    if isinstance(x, Path): return str(x)
    if isinstance(x, dict): return {str(k): to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [to_jsonable(v) for v in x]
    return x

def canonical_json(x: Any) -> str:
    return json.dumps(to_jsonable(x), ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)

def object_hash(x: Any) -> str:
    return hashlib.sha256(canonical_json(x).encode()).hexdigest()

def sha_file(p: Path) -> str:
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""): h.update(chunk)
    return h.hexdigest()

def atomic_text(path: Path, text: str) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)

def save_json(path: Path, x: Any) -> None:
    atomic_text(path, canonical_json(x))

def read_json(path: Path) -> dict:
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_csv(df: pd.DataFrame, p: Path, index: bool = False) -> None:
    atomic_text(p, df.to_csv(index=index))

def cell_ids(k, c):
    k = np.asarray(k, int); c = np.asarray(c, int)
    return np.where(k == 1, np.where(c == 1, 0, 1), np.where(c == 1, 3, 2)).astype(int)

def state_action(state):
    s = np.asarray(state, int)
    return np.where(s == 1, "retain", np.where(s == 3, "accept", "noop"))

def norm_answer(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = "".join(c for c in s if c not in set(string.punctuation))
    return " ".join(x for x in s.split() if x not in {"a", "an", "the"})

def truncate_answer(raw):
    s = str(raw).strip().split("\n")[0].strip()
    s = re.sub(r"^(answer|the answer is)\s*:?\s*", "", s, flags=re.I)
    s = s.split(". ")[0].strip().strip("\"'")
    return re.sub(r"[.,;:\s]+$", "", s).strip()

def gold_aliases(v):
    if isinstance(v, (list, tuple, np.ndarray)): items = list(v)
    else:
        try: items = json.loads(str(v))
        except Exception:
            try: items = pyast.literal_eval(str(v))
            except Exception: items = [str(v)]
    if isinstance(items, str): items = [items]
    return [str(x) for x in items if norm_answer(x)]

def load_frame(behavior_path: Path, labels_path: Path) -> pd.DataFrame:
    f = pd.read_csv(behavior_path, keep_default_na=False).reset_index(drop=True)
    lab = pd.read_parquet(labels_path)
    needed = {"question", "ctx_text", "ctx_label", "gold", "answer_surface"}
    if not needed.issubset(f.columns): raise ValueError(f"behavior columns missing: {needed-set(f.columns)}")
    lab = lab.copy(); lab["k"] = lab["kside_corr"].map({"K+": 1, "K-": 0, "K−": 0})
    if lab["k"].isna().any(): raise ValueError("알 수 없는 K 교정 라벨")
    if not lab.groupby("question")["k"].nunique().eq(1).all(): raise ValueError("질문별 K 라벨이 충돌합니다.")
    kmap = lab.drop_duplicates("question").set_index("question")["k"]
    f["k"] = f["question"].map(kmap)
    f["c"] = f["ctx_label"].map({"C+": 1, "C-": 0, "C−": 0})
    if f[["k", "c"]].isna().any().any(): raise ValueError("데이터와 라벨이 매칭되지 않습니다.")
    f[["k", "c"]] = f[["k", "c"]].astype(int)
    f["state"] = cell_ids(f.k, f.c); f["row"] = np.arange(len(f))
    f["aliases"] = f.gold.map(gold_aliases)
    assert len(f) == 1224 and f.question.nunique() == 612
    assert f.groupby("question").size().eq(2).all()
    assert f.groupby("question")["c"].nunique().eq(2).all()
    assert f.groupby("question")["k"].nunique().eq(1).all()
    assert f.groupby("state").size().reindex(range(4)).tolist() == [362, 362, 250, 250]
    return f

def expected_e14_split(frame: pd.DataFrame, fold: int, seed: int = 42) -> pd.DataFrame:
    # E14 원본의 question-wise, K-stratified outer map을 정확히 재현.
    q = frame[["question", "k"]].drop_duplicates().sort_values("question").reset_index(drop=True)
    rng = np.random.default_rng(seed); mapping = {}
    for cls in (0, 1):
        ids = q.loc[q.k.eq(cls), "question"].to_numpy().copy(); rng.shuffle(ids)
        mapping.update({s: i % 5 for i, s in enumerate(ids)})
    folds = frame.question.map(mapping).to_numpy()
    role = np.full(len(frame), "fit", object)
    role[folds == fold] = "test"; role[folds == ((fold+1) % 5)] = "cal"
    rng = np.random.default_rng(seed + 1000*fold)
    for cls in (0, 1):
        qids = np.array(sorted(set(frame.loc[(role == "cal") & frame.k.eq(cls), "question"])))
        rng.shuffle(qids); half = len(qids)//2
        role[(role == "cal") & frame.question.isin(qids[:half])] = "utility_fit"
        role[(role == "cal") & frame.question.isin(qids[half:])] = "policy_cal"
    out = pd.DataFrame({"row": frame.row, "question": frame.question, "fold": folds,
                        "role": role, "k": frame.k, "c": frame.c, "state": frame.state})
    assert out.groupby("question").role.nunique().eq(1).all()
    return out

def validate_policy(p: dict, n_heads: int = 32) -> None:
    if p.get("uses_probe_at_runtime") is not False: raise ValueError("E14 runtime policy가 아닙니다.")
    for role, sign_key in [("memory", "K_qnorm_sign"), ("context", "C_contextmass_sign"),
                           ("conflict", "Conflict_shift_sign")]:
        specs = p["role_lock"][role]; heads = [int(s["head"]) for s in specs]
        if len(heads) != len(set(heads)) or not heads: raise ValueError("중복/빈 role head")
        if not all(0 <= h < n_heads for h in heads): raise ValueError("head ID 범위 오류")
        if not all(float(s[sign_key]) in (-1., 1.) for s in specs): raise ValueError("sign 오류")
        med = np.asarray(p["normalizer"][role]["median"])
        scale = np.asarray(p["normalizer"][role]["scale"])
        assert med.shape == scale.shape == (len(heads),)
        assert np.isfinite(med).all() and np.isfinite(scale).all() and (scale > 0).all()
    assert all(np.isfinite(float(v)) for v in p["thresholds"].values())

def group_permuted_states(y, questions, seed):
    # fit 안에서만 K는 질문 간 shuffle, C는 질문별 pair swap. Test 라벨은 접근하지 않음.
    y = np.asarray(y, int); q = np.asarray(questions, str); uq = np.unique(q)
    rng = np.random.default_rng(seed)
    kq = np.array([int(y[q == x][0] in (0, 1)) for x in uq]); rng.shuffle(kq)
    out = np.empty(len(y), int)
    for x, k in zip(uq, kq):
        ix = np.flatnonzero(q == x)
        c = np.isin(y[ix], (0, 3)).astype(int)
        if rng.integers(2): c = 1-c
        out[ix] = cell_ids(np.full(len(ix), k), c)
    return out


In [ ]:
# 4. E14 원본 artifact를 읽고 split/질문/라벨/캐시 무결성을 확인
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None and not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

E14_ROOT = Path(CFG["E14_ROOT"])
ROOT = Path(CFG["OUTPUT_PARENT"]) / CFG["RUN_ID"]
assert ROOT.resolve() != E14_ROOT.resolve(), "E14 폴더에 쓰면 안 됩니다."
WORK = Path(CFG["DATA_DIR"]); WORK.mkdir(parents=True, exist_ok=True)
ROOT.mkdir(parents=True, exist_ok=True)
required = [E14_ROOT / "L13_neutral_head_bank_all_rows.npz"]
for fold in CFG["OUTER_FOLDS"]:
    required += [E14_ROOT / f"fold_{fold}" / name for name in
                 ["split_manifest.csv", "HEAD_LABELFREE_POLICY_LOCK.json", "instruction_vectors.npz"]]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("E14가 실행된 Drive 계정/폴더를 확인하세요. CFG['E14_ROOT']만 변경하면 됩니다.\n" + "\n".join(missing))

import gdown
paths = {}
for key, (fid, name) in SOURCE_FILES.items():
    p = WORK / name
    if not p.exists():
        tmp = p.with_name(p.name + ".part")
        if not gdown.download(id=fid, output=str(tmp), quiet=False): raise RuntimeError(f"download failed: {name}")
        os.replace(tmp, p)
    paths[key] = p
FRAME = load_frame(paths["behavior"], paths["klabels"])
# Router에 건네는 데이터에는 라벨/gold가 없습니다.
INPUTS = FRAME[["question", "ctx_text"]].copy()
BANK_PATH = E14_ROOT / "L13_neutral_head_bank_all_rows.npz"
with np.load(BANK_PATH, allow_pickle=False) as z: BANK = {k: z[k].copy() for k in z.files}
for key, shape in {"HEAD_Q": (1224,32,128), "HEAD_LAST": (1224,32,128),
                   "C_MASS": (1224,32), "BLOCK_LAST": (1224,4096)}.items():
    if key not in BANK or BANK[key].shape != shape: raise ValueError(f"E14 bank shape mismatch: {key}")
    if not np.isfinite(BANK[key]).all(): raise ValueError(f"non-finite bank: {key}")
assert np.array_equal(BANK["rows"], np.arange(len(FRAME)))
HQ = BANK["HEAD_Q"].astype(np.float32)
HL = BANK["HEAD_LAST"].astype(np.float32)
CM = BANK["C_MASS"].astype(np.float32)

SPLITS, E14_POLICIES, VECTOR_PATHS = {}, {}, {}
for fold in CFG["OUTER_FOLDS"]:
    folder = E14_ROOT / f"fold_{fold}"
    actual = pd.read_csv(folder / "split_manifest.csv").sort_values("row").reset_index(drop=True)
    expected = expected_e14_split(FRAME, fold, CFG["SEED"])
    for col in ["row", "question", "role", "k", "c", "state"]:
        if actual[col].tolist() != expected[col].tolist():
            raise ValueError(f"fold {fold}: E14 split과 현재 데이터 불일치: {col}")
    SPLITS[fold] = actual
    E14_POLICIES[fold] = read_json(folder / "HEAD_LABELFREE_POLICY_LOCK.json")
    validate_policy(E14_POLICIES[fold])
    VECTOR_PATHS[fold] = folder / "instruction_vectors.npz"

test_counts = np.zeros(len(FRAME), int)
for sp in SPLITS.values(): test_counts += sp.role.eq("test").to_numpy(int)
assert np.all(test_counts == 1), "각 행은 자기 outer test fold에서 정확히 1회 평가해야 합니다."
SOURCE_HASHES = {str(p.relative_to(E14_ROOT)): sha_file(p) for p in required}
SOURCE_HASHES.update({"behavior": sha_file(paths["behavior"]), "klabels": sha_file(paths["klabels"])})
SCI_CFG = {k:v for k,v in CFG.items() if k not in {"RUN_GENERATION", "RESUME"}}
RUN_MANIFEST = {"config": SCI_CFG, "source_hashes": SOURCE_HASHES, "system_prompt": SYSTEM_PROMPT,
                "protocol": "matched-role-union/readout-only/exploratory-p8s-v1",
                "analysis_software": {"numpy":np.__version__, "pandas":pd.__version__, "sklearn":sklearn.__version__},
                "note": "E14 결과를 본 후 설계한 동일 데이터셋 후속 분석; 완전히 새 blind test 아님."}
RUN_HASH = object_hash(RUN_MANIFEST)
manifest_path = ROOT / "RUN_MANIFEST.json"
if manifest_path.exists() and object_hash(read_json(manifest_path)) != RUN_HASH:
    raise RuntimeError("동일 RUN_ID의 설정/입력이 바뀌었습니다. 새 RUN_ID로 실행하여 캐시 혼합을 막으세요.")
save_json(manifest_path, RUN_MANIFEST)
save_json(ROOT/"EXECUTION_OPTIONS.json", {"RUN_GENERATION":CFG["RUN_GENERATION"],"RESUME":CFG["RESUME"]})
print("1224 rows / 612 questions / 5 folds: PASS")
print("출력:", ROOT)
display(pd.crosstab(FRAME.k, FRAME.c))
display(pd.DataFrame([{ "fold": f, **sp.role.value_counts().to_dict()} for f,sp in SPLITS.items()]))


## 5. 고정 기하학 readout

공통 vector feature는 head 수를 $H$라고 할 때
\[
x=[h_Q,\ h_{last}-h_Q,\ C_{mass}],\qquad \dim(x)=257H
\]
입니다. E14에서 사용하던 문맥 attention mass도 양쪽에 동일하게 제공합니다.
‘head vector만’이라기보다는 **동일 head의 vector + 동일 attention 통계** 비교입니다.

표준화의 평균/표준편차는 fit에서만 계산합니다.
방향 방법은 K+/K−, C+/C−의 fit 평균 차이를 정규화하여 $d_K,d_C$를 만들고,
두 평균의 중간점을 경계로 signed projection을 계산합니다.
\[
s_K=\big(z-(\mu_{K+}+\mu_{K-})/2\big)^\top d_K
\]
C도 같은 방식입니다. 이 값은 **확률이 아닙니다.**

Prototype은 fit 표준화 벡터를 단위벡터로 만든 뒤 상태별 평균 방향을 구합니다.
Test에서는 고정된 4개 prototype과 cosine similarity를 비교합니다.
이 역시 상태 label로 만든 **supervised nearest-prototype classifier**입니다.

무작위 방향·shuffled prototype은 각각 5 seed를 사용합니다.
head 선택은 고정한 조건부 대조군이며 전체 head-discovery 과정까지 무작위화한 검정은 아닙니다.

In [ ]:
# 5. 같은 role-head 집합에서 scalar / vector readout을 구성
# 핵심: 평균 방향·prototype도 supervised classification rule입니다.
# 'LR 객체를 호출하지 않음'과 'classifier가 아님'은 같지 않습니다.

def role_scores(headq, headlast, cmass, policy):
    headq = np.asarray(headq, np.float32); headlast = np.asarray(headlast, np.float32)
    cmass = np.asarray(cmass, np.float32); delta = headlast-headq
    vals = {}
    for role, sk in [("memory", "K_qnorm_sign"), ("context", "C_contextmass_sign"),
                     ("conflict", "Conflict_shift_sign")]:
        cols = []
        for s in policy["role_lock"][role]:
            h, sign = int(s["head"]), float(s[sk])
            if role == "memory": v = np.linalg.norm(headq[:,h], axis=1)
            elif role == "context": v = cmass[:,h]
            else: v = np.linalg.norm(delta[:,h],axis=1)/np.maximum(np.linalg.norm(headq[:,h],axis=1),1e-8)
            cols.append(sign*v)
        x = np.stack(cols, axis=1)
        med = np.asarray(policy["normalizer"][role]["median"], float)
        scale = np.asarray(policy["normalizer"][role]["scale"], float)
        vals[role] = np.median((x-med)/scale, axis=1)
    return np.column_stack([vals[r] for r in ["memory","context","conflict"]])

def e14_actions(scalar, policy):
    mem, ctx, cf = np.asarray(scalar).T; t = policy["thresholds"]
    retain = (mem >= t["memory_hi"]) & (cf >= t["conflict_hi"])
    accept = (~retain) & (mem <= t["memory_lo"]) & (ctx >= t["context_hi"])
    return np.where(retain, "retain", np.where(accept, "accept", "noop"))

def vector_features(headq, headlast, cmass, heads):
    # Q head vector, Q→last의 vector 변화, 같은 head의 context attention mass.
    # CMASS는 E14 scalar에도 주어졌으므로 양쪽에서 공유하는 보조 입력입니다.
    h = np.asarray(heads, int)
    q = np.asarray(headq, np.float32)[:,h,:]
    d = np.asarray(headlast, np.float32)[:,h,:]-q
    c = np.asarray(cmass, np.float32)[:,h]
    return np.concatenate([q.reshape(len(q),-1), d.reshape(len(d),-1), c], axis=1)

def per_head_scalars_from_vector(vector):
    x=np.asarray(vector, float)
    if x.ndim!=2 or x.shape[1] % 257:
        raise ValueError("Expected concat(HQ, HD, C_MASS) with head_dim=128")
    h=x.shape[1]//257
    q=x[:,:128*h].reshape(len(x),h,128)
    d=x[:,128*h:256*h].reshape(len(x),h,128)
    c=x[:,256*h:]
    qn=np.linalg.norm(q,axis=-1)
    shift=np.linalg.norm(d,axis=-1)/np.maximum(qn,1e-8)
    return np.concatenate([qn,shift,c],axis=1)

def unit_rows(x):
    x = np.asarray(x, float)
    return x/np.maximum(np.linalg.norm(x,axis=-1,keepdims=True),1e-12)

def axis_class_scores(k, c):
    k=np.asarray(k,float); c=np.asarray(c,float)
    out=np.column_stack([k+c,k-c,-k-c,-k+c])
    # K/C median rule uses >=, including exact ties; preserve that deterministic mapping.
    target=cell_ids(k>=0,c>=0)
    ties=out.argmax(1)!=target
    if np.any(ties):
        ix=np.flatnonzero(ties)
        out[ix,target[ix]]=np.nextafter(out[ix].max(1),np.inf)
    return out

def axis_scores_from_classes(s):
    # 클래스 score는 확률이라고 주장하지 않음. AUROC에 사용하는 순위 score.
    return (logsumexp(s[:,[0,1]],axis=1)-logsumexp(s[:,[2,3]],axis=1),
            logsumexp(s[:,[0,3]],axis=1)-logsumexp(s[:,[1,2]],axis=1))

@dataclass
class FrozenReadout:
    name: str
    kind: str
    feature_source: str
    params: dict
    optimizer_trained: bool = False
    supervised_geometry: bool = False

    def score(self, vector, scalar):
        x = np.asarray(scalar if self.feature_source == "scalar" else
                       per_head_scalars_from_vector(vector) if self.feature_source == "per_head_scalar" else vector, float)
        p = self.params
        if self.kind == "e14_median":
            k = x[:,0]-float(p["k_threshold"])
            c = x[:,1]-float(p["c_threshold"])
            return axis_class_scores(k, c)
        z = (x-np.asarray(p["center"]))/np.asarray(p["scale"])
        if "pca_components" in p:
            z = (z-np.asarray(p["pca_center"])) @ np.asarray(p["pca_components"]).T
        if self.kind in {"direction", "random_direction"}:
            ds = np.asarray(p["directions"])
            scores = (z @ ds.T-np.asarray(p["offsets"]))/np.asarray(p["axis_scale"])
            return axis_class_scores(scores[:,0], scores[:,1])
        if self.kind == "prototype":
            return unit_rows(z) @ np.asarray(p["prototypes"]).T
        if self.kind == "lr_reference":
            # sklearn.predict_proba를 호출하지 않아도, 이것은 학습된 LR입니다.
            return z @ np.asarray(p["weight"]).T+np.asarray(p["bias"])
        raise ValueError(self.kind)

    def as_dict(self):
        return to_jsonable({"name":self.name,"kind":self.kind,"feature_source":self.feature_source,
                            "params":self.params,"optimizer_trained":self.optimizer_trained,
                            "supervised_geometry":self.supervised_geometry})


def fit_standardizer(x):
    x = np.asarray(x, float)
    center = x.mean(axis=0); scale = np.maximum(x.std(axis=0), 1e-6)
    return {"center":center,"scale":scale}, (x-center)/scale

def fit_lr_reference(x, y, name, source, c_value, seed):
    original_x=x
    if source=="per_head_scalar": x=per_head_scalars_from_vector(x)
    p, z = fit_standardizer(x)
    model = LogisticRegression(C=c_value, max_iter=4000, solver="lbfgs", random_state=seed)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(z, np.asarray(y,int))
    if any(issubclass(w.category, ConvergenceWarning) for w in caught):
        warnings.warn(f"{name}: LR convergence warning. 출력/수렴상태를 확인하세요.")
    assert model.classes_.tolist() == [0,1,2,3]
    p.update({"weight":model.coef_,"bias":model.intercept_,"n_iter":model.n_iter_})
    out = FrozenReadout(name,"lr_reference",source,p,True,False)
    # Export한 고정 dot-product와 실제 LR logits가 같은지 확인.
    expected = model.decision_function(z)
    got = out.score(original_x if source != "scalar" else np.zeros((len(x),1)),
                    x if source == "scalar" else np.zeros((len(x),3)))
    np.testing.assert_allclose(got, expected, rtol=1e-7, atol=1e-7)
    return out

def fit_direction(x, y):
    p,z = fit_standardizer(x); y=np.asarray(y,int)
    dirs, offsets, scales = [],[],[]
    for positive in [[0,1],[0,3]]:
        mask=np.isin(y,positive)
        mu_pos=z[mask].mean(0); mu_neg=z[~mask].mean(0)
        d=unit_rows((mu_pos-mu_neg)[None])[0]
        if np.linalg.norm(d)<1e-10: raise ValueError("degenerate direction")
        off=float(((mu_pos+mu_neg)/2) @ d)
        s=z@d-off
        dirs.append(d); offsets.append(off); scales.append(max(float(s.std()),1e-6))
    p.update({"directions":np.stack(dirs),"offsets":np.array(offsets),"axis_scale":np.array(scales)})
    return FrozenReadout("direction","direction","vector",p,False,True)

def fit_prototype(x,y,name="prototype",pca_dim=None,seed=42):
    p,z=fit_standardizer(x); y=np.asarray(y,int)
    if pca_dim is not None:
        r=min(int(pca_dim),z.shape[0]-1,z.shape[1])
        pca=PCA(n_components=r,whiten=False,svd_solver="randomized",random_state=seed)
        z=pca.fit_transform(z)
        p.update({"pca_center":pca.mean_,"pca_components":pca.components_,"pca_rank":r,
                  "pca_explained_variance_sum":float(pca.explained_variance_ratio_.sum())})
    zn=unit_rows(z)
    prot=unit_rows(np.stack([zn[y==st].mean(0) for st in range(4)]))
    if not np.all(np.linalg.norm(prot,axis=1)>1e-8): raise ValueError("degenerate prototype")
    p["prototypes"]=prot
    return FrozenReadout(name,"prototype","vector",p,False,True)

def fit_random_direction(x_fit,x_pc,seed):
    p,z=fit_standardizer(x_fit); rng=np.random.default_rng(seed)
    ds=unit_rows(rng.standard_normal((2,z.shape[1])))
    zpc=(x_pc-p["center"])/p["scale"]
    offsets=np.median(zpc@ds.T,axis=0)  # 무라벨 threshold
    scales=np.maximum((z@ds.T).std(axis=0),1e-6)
    p.update({"directions":ds,"offsets":offsets,"axis_scale":scales})
    return FrozenReadout(f"random_direction_{seed}","random_direction","vector",p,False,False)


def action_margin(class_scores):
    s=np.asarray(class_scores,float)
    pred=s.argmax(axis=1)
    retain=s[:,1]-np.max(s[:,[0,2,3]],axis=1)
    accept=s[:,3]-np.max(s[:,[0,1,2]],axis=1)
    margin=np.maximum(retain,accept)
    return pred,margin

def apply_margin(class_scores,cutoff):
    pred,margin=action_margin(class_scores)
    target=state_action(pred)
    if cutoff is None: return np.full(len(pred),"noop",object)
    ok=(target!="noop") & (margin>=float(cutoff))
    return np.where(ok,target,"noop")

def cutoff_for_budget(class_scores,target_rate):
    # PC activation-derived score만 이용; test labels/metrics는 보지 않습니다.
    if target_rate<=0: return None
    _,m=action_margin(class_scores)
    return max(0.,float(np.quantile(m,1-min(float(target_rate),1.))))

def build_readouts(xfit,sfit,yfit,qfit,xpc,spc,policy,cfg,fold):
    result={}
    p={"k_threshold":float(np.median(spc[:,0])),"c_threshold":float(np.median(spc[:,1]))}
    result["e14_median4"]=FrozenReadout("e14_median4","e14_median","scalar",p)
    result["direction"]=fit_direction(xfit,yfit)
    result["prototype"]=fit_prototype(xfit,yfit)
    result["pca_prototype"]=fit_prototype(xfit,yfit,"pca_prototype",cfg["PCA_DIM"],cfg["SEED"]+fold)
    result["vector_lr"]=fit_lr_reference(xfit,yfit,"vector_lr","vector",cfg["LR_C"],cfg["SEED"]+fold)
    result["scalar_lr"]=fit_lr_reference(sfit,yfit,"scalar_lr","scalar",cfg["LR_C"],cfg["SEED"]+fold)
    result["perhead_scalar_lr"]=fit_lr_reference(xfit,yfit,"perhead_scalar_lr","per_head_scalar",cfg["LR_C"],cfg["SEED"]+fold)
    for seed in cfg["CONTROL_SEEDS"]:
        result[f"random_direction_{seed}"]=fit_random_direction(xfit,xpc,seed+1000*fold)
        # 이름은 fold 간 집계가 가능하게 원래 seed로 고정.
        result[f"random_direction_{seed}"].name=f"random_direction_{seed}"
        yp=group_permuted_states(yfit,qfit,seed+1000*fold)
        name=f"shuffled_prototype_{seed}"
        result[name]=fit_prototype(xfit,yp,name)
    return result


## 6. test 평가 전에 모든 fold의 설정 LOCK

`state_map`: 예측 K+C−→retain, K−C+→accept, 나머지→noop.
이는 **K/C에서 정한 routing target**이지 실제 QA에 최적인 action oracle이 아닙니다.

`budget`: policy_cal에서 E14 gate의 개입률을 구하고,
각 방법의 action margin 상위 부분이 비슷한 개입률을 갖도록 **PC score quantile**로 cutoff를 정합니다.
분류 label/gold/EM으로 cutoff를 탐색하지 않습니다.
Test에서 개입률이 정확히 같아지는 것은 아니며 실제 개입률을 함께 보고합니다.

PCA=32, control seed, QA 비교군은 사전 고정됩니다.
**test F1이 좋은 방법만 골라 QA를 수행하지 않습니다.**
모든 lock에 source hash와 사용 row를 기록하고, 재실행 시 같은 lock을 읽습니다.

In [ ]:
# 6. 모든 fold의 readout/threshold를 먼저 고정. 이 셀은 test 성능을 계산하지 않습니다.
FOLD_BUNDLES={}
lock_rows=[]
for fold in CFG["OUTER_FOLDS"]:
    sp=SPLITS[fold]; pol=E14_POLICIES[fold]
    fit=np.flatnonzero(sp.role.eq("fit")); pc=np.flatnonzero(sp.role.eq("policy_cal"))
    uf=np.flatnonzero(sp.role.eq("utility_fit")); test=np.flatnonzero(sp.role.eq("test"))
    assert not set(FRAME.iloc[fit].question)&set(FRAME.iloc[test].question)
    assert not set(FRAME.iloc[pc].question)&set(FRAME.iloc[test].question)
    # 동일 fold의 E14 role-head union. 다른 fold의 discovery 결과를 합치지 않음.
    heads=sorted({int(x["head"]) for specs in pol["role_lock"].values() for x in specs})
    bundle_path=ROOT/f"fold_{fold}"/"READOUT_LOCK.json"
    if CFG["RESUME"] and bundle_path.exists():
        saved=read_json(bundle_path)
        if saved["run_hash"]!=RUN_HASH or saved["heads"]!=heads:
            raise RuntimeError(f"fold {fold}: 저장한 readout과 현재 protocol이 다릅니다.")
        for key,value in [("fit_rows",fit),("pc_rows",pc),("test_rows",test)]:
            assert np.array_equal(saved[key],value), f"{key} resume mismatch"
        assert saved["source_policy_hash"]==sha_file(E14_ROOT/f"fold_{fold}"/"HEAD_LABELFREE_POLICY_LOCK.json")
        models={name:FrozenReadout(**spec) for name,spec in saved["models"].items()}
        b={k:v for k,v in saved.items() if k not in {"models","run_hash","source_policy_hash"}}
        b.update({"models":models,"fit_rows":fit,"pc_rows":pc,"test_rows":test,
                  "lock_path":bundle_path,"lock_hash":sha_file(bundle_path)})
        FOLD_BUNDLES[fold]=b
        lock_rows.append({"fold":fold,"heads":",".join(map(str,heads)),"head_count":len(heads),
                          "vector_feature_dim":b["vector_feature_dim"],"scalar_dim":3,
                          "PC_E14_intervention_rate":b["target_pc_intervention_rate"],
                          "FIT_n":len(fit),"PC_n":len(pc),"TEST_n":len(test)})
        print(f"fold {fold}: 기존 READOUT_LOCK 그대로 재사용 (재학습 없음)")
        continue
    xf=vector_features(HQ[fit],HL[fit],CM[fit],heads)
    xp=vector_features(HQ[pc],HL[pc],CM[pc],heads)
    sf=role_scores(HQ[fit],HL[fit],CM[fit],pol)
    sc=role_scores(HQ[pc],HL[pc],CM[pc],pol)
    with threadpool_limits(limits=2):
        models=build_readouts(xf,sf,FRAME.iloc[fit].state.to_numpy(),FRAME.iloc[fit].question.to_numpy(),
                              xp,sc,pol,CFG,fold)
    pc_e14=e14_actions(sc,pol)
    target_rate=float(np.mean(pc_e14!="noop"))
    margins={}
    for name,m in models.items():
        score=m.score(xp,sc)
        margins[name]={"state_map":0.,"budget":cutoff_for_budget(score,target_rate)}
        for cov in CFG["COVERAGE_GRID"]:
            margins[name][f"coverage_{cov:g}"]=cutoff_for_budget(score,cov)
    bundle={"fold":fold,"heads":heads,"policy":pol,"models":models,"margins":margins,
            "fit_rows":fit,"pc_rows":pc,"test_rows":test,"target_pc_intervention_rate":target_rate,
            "feature_description":"concat(HQ[H], (HL-HQ)[H], C_MASS[H])",
            "fit_label_source":"same outer fold fit only",
            "head_source":"same outer fold E14 role_lock (supervised utility_fit)",
            "threshold_source":"FIT labeled centroid midpoint; PC unlabeled action-margin quantile",
            "head_count":len(heads),"vector_feature_dim":int(xf.shape[1]),"scalar_dim":3,
            "label_dependency": {"head_selection":"E14 same-fold utility_fit labels",
              "geometry":"fit K/C labels", "margin_calibration":"policy_cal scores only",
              "runtime_label_input":False,"runtime_classifier_free":False,"geometry_uses_LR_optimizer":False}}
    serial={k:v for k,v in bundle.items() if k!="models"}
    serial["models"]={k:v.as_dict() for k,v in models.items()}
    serial["run_hash"]=RUN_HASH
    serial["source_policy_hash"]=sha_file(E14_ROOT/f"fold_{fold}"/"HEAD_LABELFREE_POLICY_LOCK.json")
    encoded=canonical_json(serial)
    if bundle_path.exists() and sha_file(bundle_path)!=hashlib.sha256(encoded.encode()).hexdigest():
        raise RuntimeError(f"fold {fold}: lock 재생성 결과가 다릅니다. 새 RUN_ID를 쓰세요.")
    atomic_text(bundle_path,encoded)
    bundle["lock_path"]=bundle_path; bundle["lock_hash"]=sha_file(bundle_path)
    FOLD_BUNDLES[fold]=bundle
    lock_rows.append({"fold":fold,"heads":",".join(map(str,heads)),"head_count":len(heads),
                      "vector_feature_dim":xf.shape[1],"scalar_dim":3,
                      "PC_E14_intervention_rate":target_rate,"FIT_n":len(fit),"PC_n":len(pc),"TEST_n":len(test)})
    print(f"fold {fold}: head union {heads}; dim {xf.shape[1]}; PC intervention target={target_rate:.3f}; locked")
LOCK_SUMMARY=pd.DataFrame(lock_rows)
write_csv(LOCK_SUMMARY,ROOT/"FIVEFOLD_HEAD_AND_POLICY_LOCK.csv")
save_json(ROOT/"ALL_FOLDS_LOCKED.json", {str(f):b["lock_hash"] for f,b in FOLD_BUNDLES.items()})
display(LOCK_SUMMARY)
print("모든 fold 설정 LOCK 완료. 이후 test labels는 평가용으로만 사용합니다.")


## 7–9. 캐시 기반 5-fold state/action 분석

Macro-F1은 클래스별 F1의 평균, BAcc는 클래스별 recall의 평균입니다.
4-state와 3-action의 F1은 서로 다른 과제의 점수이므로 직접 빼서 해석하지 않습니다.

AUROC는 fold별 연속 score로 계산 후 평균을 보고합니다.
서로 다른 fold에서 만든 raw score는 scale이 다를 수 있어 raw pooled AUROC를 주 지표로 쓰지 않습니다.

질문 단위 paired bootstrap은 C+/C− 쌍을 함께 재표집합니다.
CI는 **고정된 OOF 예측에 조건부**이며, head 선택/재학습 전체의 불확실성을 모두 포함하지 않습니다.
여러 보조 비교는 탐색적으로 해석하고, `prototype vs e14_median4`를 사전 지정 주 state 비교로 봅니다.

In [ ]:
# 7. 평가 함수 — state와 action을 분리. Action target은 utility oracle이 아닙니다.
def safe_auc(y,score):
    y=np.asarray(y)
    return float(roc_auc_score(y,score)) if np.unique(y).size==2 else np.nan

def classification_metrics(y,pred,labels):
    cm=confusion_matrix(y,pred,labels=labels).astype(float)
    tp=np.diag(cm); true=cm.sum(1); guessed=cm.sum(0)
    recall=np.divide(tp,true,out=np.zeros_like(tp),where=true>0)
    precision=np.divide(tp,guessed,out=np.zeros_like(tp),where=guessed>0)
    f1=np.divide(2*tp,true+guessed,out=np.zeros_like(tp),where=(true+guessed)>0)
    return {"macro_F1":float(f1.mean()),"BAcc":float(recall.mean()),
            "accuracy":float(tp.sum()/max(cm.sum(),1)),"n":int(cm.sum())},cm,precision,recall,f1

def state_report(rows,score):
    y=FRAME.iloc[rows].state.to_numpy(int); pred=np.asarray(score).argmax(1)
    out,cm,prec,rec,f1=classification_metrics(y,pred,list(range(4)))
    ks,cs=axis_scores_from_classes(np.asarray(score))
    out.update({"K_AUROC":safe_auc(FRAME.iloc[rows].k,ks),"C_AUROC":safe_auc(FRAME.iloc[rows].c,cs),
                "K_BAcc":float(balanced_accuracy_score(FRAME.iloc[rows].k,np.isin(pred,[0,1]))),
                "C_BAcc":float(balanced_accuracy_score(FRAME.iloc[rows].c,np.isin(pred,[0,3])))})
    return out,cm,prec,rec,f1,ks,cs

def action_report(rows,actions):
    truth=state_action(FRAME.iloc[rows].state.to_numpy())
    out,cm,prec,rec,f1=classification_metrics(truth,actions,ACTION_NAMES)
    out["intervention_rate"]=float(np.mean(np.asarray(actions)!="noop"))
    for j,a in enumerate(ACTION_NAMES):
        out[f"{a}_precision"]=float(prec[j]); out[f"{a}_recall"]=float(rec[j])
        out[f"{a}_F1"]=float(f1[j])
    return out,cm

def budget_actions(bundle,name,score,mode):
    return apply_margin(score,bundle["margins"][name][mode])

def infer_bundle(bundle,headq,headlast,cmass):
    # 이 함수는 label/gold/row를 받지 않음. 동일 함수가 offline과 online에서 사용됨.
    x=vector_features(headq,headlast,cmass,bundle["heads"])
    s=role_scores(headq,headlast,cmass,bundle["policy"])
    return {name:m.score(x,s) for name,m in bundle["models"].items()},s

def paired_f1_bootstrap(y, pred_a, pred_b, questions, labels, n_boot=2000, seed=42):
    # 같은 질문의 C+/C- 쌍을 함께 재표집. 이미 고정한 예측에 조건부인 CI.
    y=np.asarray(y); a=np.asarray(pred_a); b=np.asarray(pred_b); q=np.asarray(questions)
    enc={x:i for i,x in enumerate(labels)}; k=len(labels)
    yt=np.array([enc[x] for x in y]); at=np.array([enc[x] for x in a]); bt=np.array([enc[x] for x in b])
    uq, qi=np.unique(q,return_inverse=True); Q=len(uq)
    ca=np.zeros((Q,k*k)); cb=np.zeros_like(ca)
    np.add.at(ca,(qi,yt*k+at),1); np.add.at(cb,(qi,yt*k+bt),1)
    def mf(c):
        c=np.asarray(c).reshape(-1,k,k); tp=np.diagonal(c,axis1=1,axis2=2)
        den=c.sum(1)+c.sum(2)
        return np.divide(2*tp,den,out=np.zeros_like(tp),where=den>0).mean(1)
    point=float(mf(ca.sum(0))[0]-mf(cb.sum(0))[0])
    rng=np.random.default_rng(seed); ds=[]
    for start in range(0,n_boot,100):
        ix=rng.integers(Q,size=(min(100,n_boot-start),Q))
        ds.extend((mf(ca[ix].sum(1))-mf(cb[ix].sum(1))).tolist())
    lo,hi=np.quantile(ds,[.025,.975])
    return {"delta_F1":point,"CI_low":float(lo),"CI_high":float(hi),"bootstrap_questions":Q}


In [ ]:
# 8. 고정된 readout을 각 held-out test에 적용 (LLM 재실행 없음)
state_records=[]; action_records=[]; state_predictions=[]; action_predictions=[]
per_class=[]; coverage_records=[]; baseline_audit=[]
for fold,b in FOLD_BUNDLES.items():
    assert sha_file(b["lock_path"])==b["lock_hash"]
    te=b["test_rows"]
    scores,scalar=infer_bundle(b,HQ[te],HL[te],CM[te])
    folder=ROOT/f"fold_{fold}"; (folder/"confusion").mkdir(parents=True,exist_ok=True)
    truth=FRAME.iloc[te].state.to_numpy(int)
    for name,sc in scores.items():
        met,cm,prec,rec,f1,ks,cs=state_report(te,sc)
        state_records.append({"fold":fold,"method":name,**met})
        state_predictions.append(pd.DataFrame({"row":te,"fold":fold,"method":name,"true_state":truth,
                                               "pred_state":sc.argmax(1),"K_score":ks,"C_score":cs}))
        write_csv(pd.DataFrame(cm.astype(int),index=STATE_NAMES,columns=STATE_NAMES),folder/"confusion"/f"state_{name}.csv",True)
        for j,st in enumerate(STATE_NAMES):
            per_class.append({"fold":fold,"method":name,"state":st,"precision":prec[j],"recall":rec[j],
                              "F1":f1[j],"support":int(cm[j].sum())})
        for mode in ["state_map","budget"]:
            actions=budget_actions(b,name,sc,mode); am,acm=action_report(te,actions)
            method=f"{name}__{mode}"
            action_records.append({"fold":fold,"method":method,**am})
            action_predictions.append(pd.DataFrame({"row":te,"fold":fold,"method":method,
                                                     "target_action":state_action(truth),"action":actions}))
            write_csv(pd.DataFrame(acm.astype(int),index=ACTION_NAMES,columns=ACTION_NAMES),folder/"confusion"/f"action_{method}.csv",True)
        if name in ["direction","prototype","pca_prototype","vector_lr","scalar_lr","perhead_scalar_lr"]:
            for cov in CFG["COVERAGE_GRID"]:
                acts=budget_actions(b,name,sc,f"coverage_{cov:g}")
                am,_=action_report(te,acts)
                coverage_records.append({"fold":fold,"method":name,"PC_target_coverage":cov,**am})
    # 원래 E14의 3-action 정책은 median4 진단과 별개.
    orig=e14_actions(scalar,b["policy"])
    for name,acts in [("e14_rule",orig),("all_noop",np.full(len(te),"noop",object))]:
        am,_=action_report(te,acts)
        action_records.append({"fold":fold,"method":name,**am})
        action_predictions.append(pd.DataFrame({"row":te,"fold":fold,"method":name,
                                               "target_action":state_action(truth),"action":acts}))
    saved=E14_ROOT/f"fold_{fold}"/"test_head_label_free_gate.csv"
    if saved.exists():
        old=pd.read_csv(saved).set_index("row").loc[te]
        old_a=old.action.map({"retain_mid":"retain","accept_mid":"accept","noop":"noop"})
        if old_a.isna().any(): raise ValueError("E14 saved action format mismatch")
        disagreements=int(np.sum(orig!=old_a.to_numpy()))
        baseline_audit.append({"fold":fold,"n":len(te),"cached_vs_saved_E14_action_mismatch":disagreements})
        if disagreements: warnings.warn(f"fold {fold}: 기존 저장 action과 bank 재계산이 {disagreements}행 다릅니다. QA 단계 parity를 확인하세요.")
    print("offline held-out fold",fold,"done")

STATE_FOLDS=pd.DataFrame(state_records); ACTION_FOLDS=pd.DataFrame(action_records)
STATE_OOF=pd.concat(state_predictions,ignore_index=True).sort_values(["method","row"]).reset_index(drop=True)
ACTION_OOF=pd.concat(action_predictions,ignore_index=True).sort_values(["method","row"]).reset_index(drop=True)
for df in [STATE_OOF,ACTION_OOF]:
    assert df.groupby("method").size().eq(1224).all()
    assert df.groupby("method").row.nunique().eq(1224).all()
state_pooled=[]; action_pooled=[]
for name,g in STATE_OOF.groupby("method"):
    met,cm,_,_,_=classification_metrics(g.true_state,g.pred_state,list(range(4)))
    ff=STATE_FOLDS[STATE_FOLDS.method.eq(name)]
    state_pooled.append({"method":name,"OOF_macro_F1":met["macro_F1"],"OOF_BAcc":met["BAcc"],
                         "OOF_accuracy":met["accuracy"],"fold_F1_mean":ff.macro_F1.mean(),
                         "fold_F1_std":ff.macro_F1.std(ddof=1),"K_AUROC_fold_mean":ff.K_AUROC.mean(),
                         "C_AUROC_fold_mean":ff.C_AUROC.mean(),"n":met["n"]})
    write_csv(pd.DataFrame(cm.astype(int),index=STATE_NAMES,columns=STATE_NAMES),ROOT/"confusion"/f"OOF_state_{name}.csv",True)
for name,g in ACTION_OOF.groupby("method"):
    met,cm,prec,rec,f1=classification_metrics(g.target_action,g.action,ACTION_NAMES)
    action_pooled.append({"method":name,**met,"intervention_rate":g.action.ne("noop").mean(),
                          **{f"{a}_recall":rec[j] for j,a in enumerate(ACTION_NAMES)},
                          **{f"{a}_precision":prec[j] for j,a in enumerate(ACTION_NAMES)}})
    write_csv(pd.DataFrame(cm.astype(int),index=ACTION_NAMES,columns=ACTION_NAMES),ROOT/"confusion"/f"OOF_action_{name}.csv",True)
STATE_SUMMARY=pd.DataFrame(state_pooled); ACTION_SUMMARY=pd.DataFrame(action_pooled)
for df,name in [(STATE_FOLDS,"STATE_FOLD_METRICS"),(ACTION_FOLDS,"ACTION_FOLD_METRICS"),
                (STATE_OOF,"STATE_OOF_PREDICTIONS"),(ACTION_OOF,"ACTION_OOF_PREDICTIONS"),
                (STATE_SUMMARY,"STATE_OOF_SUMMARY"),(ACTION_SUMMARY,"ACTION_OOF_SUMMARY"),
                (pd.DataFrame(per_class),"STATE_PER_CLASS"),(pd.DataFrame(coverage_records),"COVERAGE_DIAGNOSTIC"),
                (pd.DataFrame(baseline_audit),"E14_REPLAY_AUDIT")]: write_csv(df,ROOT/(name+".csv"))
PRIMARY_NAMES=["e14_median4","scalar_lr","perhead_scalar_lr","direction","prototype","pca_prototype","vector_lr"]
print("4-state: 동일 fold별 role-head union / 같은 test. LR 수치는 새로 측정하며 기존 0.6306을 대입하지 않습니다.")
display(STATE_SUMMARY[STATE_SUMMARY.method.isin(PRIMARY_NAMES)].sort_values("method"))
print("Action target = K+C-:retain / K-C+:accept / 나머지:noop. 실제 최적 개입의 정답은 아닙니다.")
display(ACTION_SUMMARY[ACTION_SUMMARY.method.isin(["e14_rule","all_noop","direction__budget","prototype__budget", "prototype__state_map","vector_lr__budget","scalar_lr__budget"])])


In [ ]:
# 9. paired comparison: 질문 단위 bootstrap, seed controls, fold별 차이
comparisons=[]
for first,ref in [("prototype","e14_median4"),("direction","e14_median4"),
                  ("pca_prototype","e14_median4"),("vector_lr","scalar_lr"),("vector_lr","perhead_scalar_lr"),
                  ("perhead_scalar_lr","scalar_lr"),
                  ("prototype","vector_lr")]:
    a=STATE_OOF[STATE_OOF.method.eq(first)].sort_values("row")
    b=STATE_OOF[STATE_OOF.method.eq(ref)].sort_values("row")
    assert np.array_equal(a.row,b.row) and np.array_equal(a.true_state,b.true_state)
    stats=paired_f1_bootstrap(a.true_state.to_numpy(),a.pred_state.to_numpy(),b.pred_state.to_numpy(),
                              FRAME.iloc[a.row.to_numpy()].question.to_numpy(),list(range(4)),CFG["N_BOOT"],CFG["SEED"])
    f=STATE_FOLDS.pivot(index="fold",columns="method",values="macro_F1")
    comparisons.append({"method":first,"reference":ref,"folds_improved":int((f[first]>f[ref]).sum()),**stats})
PAIRED_STATE=pd.DataFrame(comparisons)
write_csv(PAIRED_STATE,ROOT/"PAIRED_STATE_F1_BOOTSTRAP.csv")
display(PAIRED_STATE)
control_rows=[]
for prefix in ["random_direction_","shuffled_prototype_"]:
    g=STATE_SUMMARY[STATE_SUMMARY.method.str.startswith(prefix)]
    control_rows.append({"control":prefix.rstrip("_"),"seeds":len(g),
                         "OOF_F1_seed_mean":g.OOF_macro_F1.mean(),"OOF_F1_seed_std":g.OOF_macro_F1.std(ddof=1),
                         "note":"heads는 고정; readout만 random/shuffle한 conditional control"})
CONTROL_SUMMARY=pd.DataFrame(control_rows)
write_csv(CONTROL_SUMMARY,ROOT/"CONTROL_SEED_SUMMARY.csv")
display(CONTROL_SUMMARY)
print("주의: test 결과를 보고 generation할 방법이나 threshold를 바꾸지 않습니다. QA_METHODS는 1절에서 고정됐습니다.")
print("이 CI는 이미 학습·선택된 예측에 대한 질문 재표집 CI이며, 전체 재학습/선택 불확실성을 포함하지 않습니다.")


## 10–12. 실제 Llama runtime

여기서부터 `RUN_GENERATION=True`일 때만 GPU가 필요합니다.
E14와 같은 Llama-3.1-8B-Instruct, Q→Context prompt, 192 new tokens를 사용합니다.
`L13`은 기존 E14의 **`model.layers[13]` (0-based)**을 뜻합니다.

**한 번의 prefill에서** L13 head를 읽어 action을 정하고 L14–18 residual에 개입합니다.
이후에는 같은 KV cache로 autoregressive decoding을 이어갑니다.
‘전체 답변을 한 번의 forward로 생성한다’는 뜻이 아닙니다.
Action은 prefill에서 한 번 고정하고, E14처럼 decode step에도 같은 선택에 따른 개입을 유지합니다.

E14의 **scaled instruction vector × ALPHA=0.5, 상대 norm cap=0.05**를 유지합니다.
E15의 `delta=rho*norm(hidden)*unit_vector`로 바꾸지 않습니다.
head readout과 steering 강도를 동시에 바꾸면 원인 비교가 어려워지기 때문입니다.

모델 SHA와 runtime bank 일치 여부를 fit 샘플로 점검합니다.
검사 실패 시 threshold를 임의로 바꾸지 말고 모델 revision/입력/bank를 확인합니다.
Capture hook 오버헤드를 모든 조건에 공통으로 포함하며 ‘LR를 빼니 더 빠르다’를 사전 가정하지 않습니다.

### 실행 오류 수정: `e17-parity-layout-r1`

**이미 1~11번을 실행하고 12번에서 멈춘 런타임:** 수정된 **12번 → 13번 이후**만 실행합니다.
모델/Probe 재학습, activation 재수집, 결과 삭제는 필요 없습니다. 새 런타임이면 처음부터 실행하며 기존 lock을 재사용합니다.

12번은 이전 검사와 **같은 8개 fit 행**을 E14 bank를 수집한 연속 4행 batch·padding으로 재현합니다.
원래 batch의 동반 행은 frozen forward shape 재현에만 사용하고 라벨/gold/상태 예측 평가에는 사용하지 않습니다.
`use_cache=False/True`도 같은 입력으로 비교하고, **기존 오차 한도 0.02 / 0.01은 유지**합니다.
다른 batch로 재구성했을 때의 수치 및 action 변화는 `FIT_BATCH_SENSITIVITY.csv`,
`FIT_BATCH_ROUTE_SENSITIVITY.csv`로 별도 기록합니다. 동일-layout 검사에 실패하면 QA는 계속 차단합니다.

수정 범위는 검사/재개 안내 및 QA 시작 전 검증입니다. `CFG`, E14 bank, readout, head set,
threshold, steering, 기존 분석 결과, generation의 batching/계산 코드는 변경하지 않았습니다.


In [ ]:
# 10. GPU 준비 (RUN_GENERATION=False이면 건너뜀)
MODEL=None; TOK=None; REVISION=None
if CFG["RUN_GENERATION"]:
    import torch, transformers
    from getpass import getpass
    from huggingface_hub import get_token, login, model_info
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from transformers.models.llama.modeling_llama import apply_rotary_pos_emb, repeat_kv
    if transformers.__version__ != CFG["TRANSFORMERS_VERSION"]:
        raise RuntimeError("transformers 이전 버전이 메모리에 남았습니다. 세션 재시작 후 처음부터 실행하세요.")
    if not torch.cuda.is_available(): raise RuntimeError("실제 QA generation에는 GPU가 필요합니다. 캐시 분석만 하려면 RUN_GENERATION=False.")
    torch.manual_seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
    gc.collect(); torch.cuda.empty_cache()
    free,total=torch.cuda.mem_get_info()
    print("GPU:",torch.cuda.get_device_name(0),f"free={free/2**30:.1f} GiB / total={total/2**30:.1f} GiB")
    if free < 19*2**30: raise RuntimeError("BF16 8B 모델을 위한 여유 GPU 메모리가 부족합니다. A100 런타임을 사용하거나 기존 모델 세션을 종료하세요.")
    token=os.environ.get("HF_TOKEN") or get_token()
    if not token:
        try:
            from google.colab import userdata
            token=userdata.get("HF_TOKEN")
        except Exception: pass
    if not token: token=getpass("Hugging Face Read token: ").strip()
    login(token=token,add_to_git_credential=False)
    revision_source="override"
    REVISION=CFG["MODEL_REVISION_OVERRIDE"].strip()
    if not REVISION:
        rp=Path(CFG["MODEL_REVISION_FILE"])
        if rp.exists(): REVISION=read_json(rp)["revision"]; revision_source=str(rp)
        else:
            REVISION=model_info(CFG["MODEL_ID"],token=token).sha
            revision_source="resolved_now; E14 revision metadata unavailable; bank parity required"
            warnings.warn("E14/E11 revision 파일이 없어 현재 SHA를 고정했습니다. 아래 bank parity 실패 시 진행하지 않습니다.")
    model_metadata={"model":CFG["MODEL_ID"],"revision":REVISION,"revision_source":revision_source,
                    "transformers":transformers.__version__,"dtype":"bfloat16","attention":"sdpa"}
    p=ROOT/"MODEL_LOCK.json"
    if p.exists() and read_json(p)!=model_metadata: raise RuntimeError("기존 E17 실행과 model revision이 다릅니다. 새 RUN_ID 필요.")
    save_json(p,model_metadata)
    TOK=AutoTokenizer.from_pretrained(CFG["MODEL_ID"],revision=REVISION,token=token,use_fast=True)
    TOK.padding_side="left"
    if TOK.pad_token_id is None: TOK.pad_token=TOK.eos_token
    MODEL=AutoModelForCausalLM.from_pretrained(CFG["MODEL_ID"],revision=REVISION,token=token,
            dtype=torch.bfloat16,device_map={"":"cuda:0"},attn_implementation="sdpa",trust_remote_code=False).eval()
    for p in MODEL.parameters(): p.requires_grad_(False)
    assert MODEL.config.num_attention_heads==32 and MODEL.config.hidden_size==4096
    assert len(MODEL.model.layers)==32
    N_HEADS=MODEL.config.num_attention_heads; N_KV_HEADS=MODEL.config.num_key_value_heads
    HEAD_DIM=MODEL.config.hidden_size//N_HEADS
    del token
    print("Pinned model loaded:",REVISION)
    print(f"QA 상한: {len(QA_METHODS)}조건 × 1224행. 답변당 최대 192토큰; 동일 RUN_ID는 완료 batch 재사용.")
else:
    print("GPU generation 생략: CPU state/routing 결과는 이미 저장되었습니다.")


In [ ]:
# 11. 실제 one-prefill runtime: L13 read → 고정 readout → L14~18 steering
# inference 함수에 K/C/gold를 전달하지 않습니다. 질문/문맥만 pack합니다.
@dataclass(frozen=True)
class PromptInput:
    question: str
    context: str

def render_prompt(item):
    user="Question: "+item.question+"\nContext: "+item.context+"\nAnswer:"
    text=TOK.apply_chat_template([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user}],
                                 tokenize=False,add_generation_prompt=True)
    base=text.rfind(user)
    if base<0: raise RuntimeError("prompt 위치를 찾을 수 없습니다.")
    qs=base+len("Question: "); qe=qs+len(item.question)
    cs=base+len("Question: ")+len(item.question)+len("\nContext: "); ce=cs+len(item.context)
    enc=TOK(text,add_special_tokens=False,return_offsets_mapping=True)
    qpos=[i for i,(s,e) in enumerate(enc["offset_mapping"]) if e>qs and s<qe and e>s]
    cpos=[i for i,(s,e) in enumerate(enc["offset_mapping"]) if e>cs and s<ce and e>s]
    if not qpos or not cpos: raise RuntimeError("Q/Context token span resolution failed")
    if max(qpos)>=min(cpos): raise RuntimeError("Q-first causal ordering 위반")
    return {"ids":enc["input_ids"],"qpos":qpos,"cpos":cpos}

def pack_inputs(rows):
    items=[PromptInput(str(INPUTS.iloc[int(i)].question),str(INPUTS.iloc[int(i)].ctx_text)) for i in rows]
    packed=[render_prompt(x) for x in items]
    T=max(len(x["ids"]) for x in packed)
    if T>CFG["MAX_PROMPT_TOKENS"]: raise RuntimeError(f"prompt too long: {T}; silent truncation 금지")
    dev=next(MODEL.parameters()).device; B=len(rows)
    ids=torch.full((B,T),TOK.pad_token_id,dtype=torch.long,device=dev); mask=torch.zeros_like(ids)
    qidx=[]; cposs=[]
    for j,p in enumerate(packed):
        n=len(p["ids"]); off=T-n
        ids[j,off:]=torch.tensor(p["ids"],device=dev); mask[j,off:]=1
        qidx.append(off+p["qpos"][-1]); cposs.append([off+i for i in p["cpos"]])
    pos=(mask.cumsum(-1)-1).clamp_min(0)
    return ids,mask,pos,torch.tensor(qidx,device=dev),cposs

class HeadCapture:
    def __init__(self,qidx,cposs,mask,pos):
        self.qidx=qidx; self.cposs=cposs; self.mask=mask; self.pos=pos
        self.phase="prefill"; self.handles=[]; self.headq=None; self.headlast=None; self.cmass=None
    def __enter__(self):
        attn=MODEL.model.layers[CFG["READ_LAYER"]].self_attn
        def attn_pre(mod,args,kwargs):
            if self.phase!="prefill": return
            h=kwargs.get("hidden_states",args[0] if args else None)
            if h is None: raise RuntimeError("attention hidden_states missing")
            B,T,_=h.shape
            q=mod.q_proj(h).view(B,T,N_HEADS,HEAD_DIM).transpose(1,2)
            k=mod.k_proj(h).view(B,T,N_KV_HEADS,HEAD_DIM).transpose(1,2)
            pe=kwargs.get("position_embeddings")
            cos,sin=MODEL.model.rotary_emb(h,self.pos) if pe is None else pe
            q,k=apply_rotary_pos_emb(q,k,cos,sin); k=repeat_kv(k,N_HEADS//N_KV_HEADS)
            scores=torch.einsum("bhd,bhkd->bhk",q[:,:,-1,:].float(),k.float())*(HEAD_DIM**-.5)
            prob=torch.softmax(scores.masked_fill(self.mask[:,None,:].eq(0),-1e9),dim=-1)
            cm=torch.zeros((B,N_HEADS),device=h.device)
            for j in range(B): cm[j]=prob[j,:,self.cposs[j]].sum(-1)
            self.cmass=cm.detach().float().cpu().numpy()
        def oproj_pre(mod,args):
            if self.phase!="prefill": return
            x=args[0]; B,T,_=x.shape; hv=x.view(B,T,N_HEADS,HEAD_DIM)
            self.headq=hv[torch.arange(B,device=x.device),self.qidx].detach().float().cpu().numpy()
            self.headlast=hv[:,-1].detach().float().cpu().numpy()
        self.handles.append(attn.register_forward_pre_hook(attn_pre,with_kwargs=True))
        self.handles.append(attn.o_proj.register_forward_pre_hook(oproj_pre))
        return self
    def __exit__(self,*args):
        for h in self.handles: h.remove()
        self.handles=[]


def choose_runtime(bundle,method,headq,headlast,cmass):
    # row ID / 라벨 / 정답 없이 내부 activation 값만 입력합니다.
    base_method=method.removesuffix("_random_steer")
    scalar=role_scores(headq,headlast,cmass,bundle["policy"])
    if base_method=="e14_rule":
        return e14_actions(scalar,bundle["policy"]),None,None
    name,mode=base_method.split("__",1)
    x=vector_features(headq,headlast,cmass,bundle["heads"])
    score=bundle["models"][name].score(x,scalar)
    pred,margin=action_margin(score)
    actions=budget_actions(bundle,name,score,mode)
    return actions,pred,margin

class RuntimeSession(HeadCapture):
    def __init__(self,bundle,method,qidx,cposs,mask,pos,vectors):
        super().__init__(qidx,cposs,mask,pos)
        self.bundle=bundle; self.method=method; self.vectors=vectors
        self.actions=np.full(len(qidx),"noop",object)
        self.pred=None; self.margin=None; self.route_ms=0.; self.read_calls=0
        self.max_ratio=np.zeros(len(qidx)); self.first_action_set=False
    def __enter__(self):
        super().__enter__()
        readblock=MODEL.model.layers[CFG["READ_LAYER"]]
        def decide(mod,args,out):
            if self.phase!="prefill": return out
            self.read_calls+=1
            t0=time.perf_counter()
            if self.method=="noop": self.actions[:]="noop"
            elif self.method=="always_retain": self.actions[:]="retain"
            elif self.method=="always_accept": self.actions[:]="accept"
            else: self.actions,self.pred,self.margin=choose_runtime(self.bundle,self.method,self.headq,self.headlast,self.cmass)
            self.route_ms+=(time.perf_counter()-t0)*1000
            self.first_action_set=True
            return out
        self.handles.append(readblock.register_forward_hook(decide))
        for layer in CFG["WRITE_LAYERS"]:
            def steer(mod,args,out,ll=layer):
                if not self.first_action_set: raise RuntimeError("read-before-write 순서 위반")
                h=out[0] if isinstance(out,tuple) else out; last=h[:,-1,:]
                delta=torch.zeros_like(last)
                v=torch.as_tensor(self.vectors[ll],device=h.device,dtype=h.dtype)
                for j,a in enumerate(self.actions):
                    if a!="noop": delta[j]=(1. if a=="accept" else -1.)*CFG["ALPHA"]*v
                if not torch.any(delta): return out
                # E14와 같은 scaled instruction vector + norm cap; E15 방식으로 바꾸지 않음.
                maxnorm=CFG["MAX_DELTA_RATIO"]*last.float().norm(dim=-1,keepdim=True)
                scale=(maxnorm/delta.float().norm(dim=-1,keepdim=True).clamp_min(1e-12)).clamp(max=1).to(h.dtype)
                applied=delta*scale
                ratios=applied.float().norm(dim=-1)/last.float().norm(dim=-1).clamp_min(1e-12)
                self.max_ratio=np.maximum(self.max_ratio,ratios.detach().cpu().numpy())
                new=h.clone(); new[:,-1,:]=last+applied
                return (new,)+out[1:] if isinstance(out,tuple) else new
            self.handles.append(MODEL.model.layers[layer].register_forward_hook(steer))
        return self


def load_vectors(fold,random_direction=False):
    with np.load(VECTOR_PATHS[fold],allow_pickle=False) as z:
        v={l:z[f"L{l}"].astype(np.float32) for l in CFG["WRITE_LAYERS"]}
    for l,a in v.items():
        assert a.shape==(4096,) and np.isfinite(a).all()
        if np.linalg.norm(a)<1e-10: raise ValueError(f"zero steering vector L{l}")
    if random_direction:
        rng=np.random.default_rng(CFG["SEED"]+91000+fold)
        v={l:(unit_rows(rng.standard_normal((1,len(a))))[0]*np.linalg.norm(a)).astype(np.float32) for l,a in v.items()}
    return v


def generate_one_batch(rows,bundle,method,vectors):
    with torch.inference_mode():
        ids,mask,pos,qidx,cposs=pack_inputs(rows); B=len(rows)
        eos={TOK.eos_token_id}; eot=TOK.convert_tokens_to_ids("<|eot_id|>")
        if isinstance(eot,int) and eot>=0 and eot!=TOK.unk_token_id: eos.add(eot)
        eos.discard(None); eosids=torch.tensor(sorted(eos),device=ids.device)
        ended=torch.zeros(B,dtype=torch.bool,device=ids.device)
        lengths=torch.zeros(B,dtype=torch.long,device=ids.device); pieces=[]
        session=RuntimeSession(bundle,method,qidx,cposs,mask,pos,vectors)
        torch.cuda.synchronize(); t0=time.perf_counter()
        with session:
            output=MODEL.model(input_ids=ids,attention_mask=mask,position_ids=pos,
                    cache_position=torch.arange(ids.shape[1],device=ids.device),use_cache=True,return_dict=True)
            past=output.past_key_values; logits=MODEL.lm_head(output.last_hidden_state[:,-1,:])
            for step in range(CFG["MAX_NEW_TOKENS"]):
                nxt=logits.argmax(-1); nxt=torch.where(ended,torch.full_like(nxt,TOK.pad_token_id),nxt)
                active=~ended; lengths+=active.long(); pieces.append(nxt.detach().cpu())
                ended|=torch.isin(nxt,eosids)&active
                if bool(ended.all()) or step+1==CFG["MAX_NEW_TOKENS"]: break
                session.phase="decode"
                mask=torch.cat([mask,torch.ones((B,1),device=mask.device,dtype=mask.dtype)],dim=-1)
                dpos=(mask.sum(-1)-1)[:,None]
                output=MODEL.model(input_ids=nxt[:,None],attention_mask=mask,position_ids=dpos,
                    cache_position=torch.tensor([mask.shape[1]-1],device=ids.device),
                    past_key_values=past,use_cache=True,return_dict=True)
                past=output.past_key_values; logits=MODEL.lm_head(output.last_hidden_state[:,-1,:])
        torch.cuda.synchronize(); elapsed=time.perf_counter()-t0
        assert session.read_calls==1, "prefill read는 배치당 정확히 한 번이어야 합니다."
        assert float(session.max_ratio.max()) <= CFG["MAX_DELTA_RATIO"]*1.025+1e-5
        texts=TOK.batch_decode(torch.stack(pieces,1),skip_special_tokens=True)
        result=[]
        for j,(r,text) in enumerate(zip(rows,texts)):
            result.append({"row":int(r),"fold":bundle["fold"],"method":method,"raw":text,
                "action":str(session.actions[j]),"pred_state":int(session.pred[j]) if session.pred is not None else None,
                "margin":float(session.margin[j]) if session.margin is not None else None,
                "seconds_per_row_batch_average":float(elapsed/B),"route_ms_batch":float(session.route_ms),
                "generated_tokens":int(lengths[j].item()),"max_delta_ratio":float(session.max_ratio[j])})
        return result


def cached_generate(rows,bundle,method):
    rows=list(map(int,rows)); folder=ROOT/f"fold_{bundle['fold']}"/"generation"/method
    folder.mkdir(parents=True,exist_ok=True)
    fingerprint=object_hash({"run":RUN_HASH,"readout_lock":bundle["lock_hash"],"method":method,
                             "revision":REVISION,"vectors":sha_file(VECTOR_PATHS[bundle["fold"]]),"rows":rows})
    header={"fingerprint":fingerprint,"rows":rows,"method":method}
    mp=folder/"CACHE_MANIFEST.json"
    if mp.exists() and read_json(mp)!=header: raise RuntimeError("generation cache mismatch; new RUN_ID required")
    save_json(mp,header)
    vectors=load_vectors(bundle["fold"],method.endswith("_random_steer"))
    records=[]
    for st in range(0,len(rows),CFG["BATCH_SIZE"]):
        ix=rows[st:st+CFG["BATCH_SIZE"]]; p=folder/f"batch_{st:06d}.json"
        if CFG["RESUME"] and p.exists():
            cached=read_json(p)
            if cached["fingerprint"]!=fingerprint or cached["rows"]!=ix: raise RuntimeError("batch cache identity mismatch")
            result=cached["records"]
        else:
            result=generate_one_batch(ix,bundle,method,vectors)
            save_json(p,{"fingerprint":fingerprint,"rows":ix,"records":result})
        assert [x["row"] for x in result]==ix
        records+=result
        if st % (CFG["BATCH_SIZE"]*20)==0: print("fold",bundle["fold"],method,min(st+len(ix),len(rows)),"/",len(rows),flush=True)
    df=pd.DataFrame(records).sort_values("row").reset_index(drop=True)
    df=df[["row","fold","method","raw","action","pred_state","margin",
           "seconds_per_row_batch_average","route_ms_batch","generated_tokens","max_delta_ratio"]]
    assert df.row.tolist()==sorted(rows) and df.row.nunique()==len(rows)
    assert sha_file(bundle["lock_path"])==bundle["lock_hash"]
    return df


In [ ]:
# 12. Bank/runtime parity — E14 원래 batch를 재현하고 regrouping 차이는 분리
# PATCH e17-parity-layout-r1. 과학 설정/학습된 readout/threshold는 변경하지 않습니다.
# E14 collect_bank(np.arange(n))는 연속 4행을 left-padding하여 저장했습니다.
# 이전 검사는 state별 2행을 재조합하여 다른 GEMM/SDPA shape와 padding을 비교했습니다.
E17_PARITY_PATCH = "e17-parity-layout-r1"
E14_BANK_BATCH_SIZE = 4  # 확인한 E14 원본의 수집 설정. 오차를 보고 선택하는 값이 아닙니다.
BANK_PARITY_APPROVAL = None


def _select_e14_fit_batches(fit_rows, states, n_rows, batch_size=4):
    """이전 검사와 같은 state별 2개 fit 행을 원래 bank batch에 되돌립니다.

    동반 행의 라벨/gold는 사용하지 않습니다. 동반 행은 같은 batch shape/padding을
    재현하는 frozen forward용이며, 비교·readout 진단은 선택한 fit 행에서만 수행합니다.
    """
    fit = np.asarray(fit_rows, int)
    selected = []
    for state in range(4):
        rows = fit[states[fit] == state][:2]
        if len(rows) < 2:
            raise RuntimeError(f"fit parity 표본 부족: state={state}")
        selected.extend(rows.tolist())
    targets = np.asarray(selected, int)
    starts = sorted({int(row) // batch_size * batch_size for row in targets})
    batches = [np.arange(st, min(st + batch_size, n_rows), dtype=int) for st in starts]
    assert set(targets).issubset(set(fit))
    return batches, targets


def _capture_parity_batch(rows, use_cache=False):
    ids, mask, pos, qidx, cposs = pack_inputs(rows)
    block_value = {}
    def save_block(mod, args, out):
        h = out[0] if isinstance(out, tuple) else out
        block_value["BLOCK_LAST"] = h[:, -1, :].detach().float().cpu().numpy().copy()
    handle = MODEL.model.layers[CFG["READ_LAYER"]].register_forward_hook(save_block)
    try:
        with torch.inference_mode(), HeadCapture(qidx, cposs, mask, pos) as cap:
            out = MODEL.model(input_ids=ids, attention_mask=mask, position_ids=pos,
                              use_cache=use_cache, return_dict=True)
        got = {"HEAD_Q": cap.headq.copy(), "HEAD_LAST": cap.headlast.copy(),
               "C_MASS": cap.cmass.copy(), **block_value}
        del out
    finally:
        handle.remove()
    for key, value in got.items():
        if not np.isfinite(value).all():
            raise RuntimeError(f"non-finite live activation: {key}")
    return got, int(ids.shape[1])


def _parity_rows(stage, rows, live, reference, padded_length, blocking):
    rec = []
    for key, value in live.items():
        old = np.asarray(reference[key], dtype=np.float32)
        value = np.asarray(value, dtype=np.float32)
        if value.shape != old.shape:
            raise RuntimeError(f"parity shape mismatch: {key}: {value.shape} != {old.shape}")
        if not np.isfinite(old).all():
            raise RuntimeError(f"non-finite reference activation: {key}")
        rel = float(np.linalg.norm(value - old) / max(np.linalg.norm(old), 1e-8))
        absolute = float(np.max(np.abs(value - old)))
        ok = (absolute <= CFG["MASS_ABS_ERROR_MAX"] if key == "C_MASS"
              else rel <= CFG["BANK_REL_ERROR_MAX"])
        rec.append({"stage": stage, "first_row": int(rows[0]), "n": len(rows),
                    "rows": ",".join(map(str, rows)), "padded_length": padded_length,
                    "key": key, "relative_error": rel, "max_abs_error": absolute,
                    "within_original_tolerance": bool(ok), "blocking": bool(blocking),
                    "pass": bool(ok) if blocking else None})
    return rec


def _run_e14_parity():
    fit = np.asarray(FOLD_BUNDLES[0]["fit_rows"], int)
    states = FRAME.state.to_numpy(int)
    batches, targets = _select_e14_fit_batches(fit, states, len(INPUTS), E14_BANK_BATCH_SIZE)
    target_set = set(map(int, targets))
    records = []
    current_by_row = {}
    print("이전 검사와 같은 fit 행:", targets.tolist())
    print("E14 원래 수집 batch를 그대로 재현:", [ix.tolist() for ix in batches])
    print("동반 행은 frozen forward의 batch 재현용입니다. 라벨/gold/예측 평가는 fit 검사 행만 사용합니다.")
    print("검사 기준 유지:", CFG["BANK_REL_ERROR_MAX"], "/", CFG["MASS_ABS_ERROR_MAX"])
    for ix in batches:
        keep = np.array([int(r) in target_set for r in ix], bool)
        checked = ix[keep]
        assert set(checked).issubset(set(fit)), "parity 비교 대상은 fit 행이어야 합니다."
        captured, length = _capture_parity_batch(ix, use_cache=False)
        live = {key: value[keep] for key, value in captured.items()}
        reference = {key: BANK[key][checked] for key in live}
        rec = _parity_rows("bank_same_layout", checked, live, reference, length, True)
        for j, row in enumerate(checked):
            current_by_row[int(row)] = {key: val[j].copy() for key, val in live.items()}
        # QA의 use_cache=True prefill도 동일 입력에서 별도로 확인합니다.
        captured_cache, cached_length = _capture_parity_batch(ix, use_cache=True)
        cached_live = {key: value[keep] for key, value in captured_cache.items()}
        rec += _parity_rows("prefill_cache_on_vs_off", checked, cached_live, live, cached_length, True)
        for r in rec:
            r["original_input_batch_rows"] = ",".join(map(str, ix))
        records += rec
    parity = pd.DataFrame(records)
    write_csv(parity, ROOT / "BANK_RUNTIME_PARITY.csv")
    display(parity)
    if not parity["within_original_tolerance"].all():
        # 안전 검사는 끄지 않습니다. 동일 batch에서도 실패하면 다른 원인이 남아 있습니다.
        raise RuntimeError(
            "동일 batch/padding에서도 bank parity가 실패했습니다. QA를 중단합니다. "
            "BANK_RUNTIME_PARITY.csv의 stage/key와 모델 SHA·torch/CUDA·GPU를 확인하세요. "
            "threshold를 올리거나 기존 bank/readout을 덮어쓰지 않습니다.")

    # 원래 bank와 runtime의 batch regrouping은 동일성 검사와 별개인 수치 민감도 진단.
    # 위의 원본-layout 및 cache-on 검사가 모두 통과해야만 이 단계에 도달합니다.
    rows = targets
    regrouped = targets.copy()  # 이전 실패 검사와 동일한 state별 묶음/순서
    sensitivity, route_checks = [], []
    bundle = FOLD_BUNDLES[0]
    for start in range(0, len(regrouped), CFG["BATCH_SIZE"]):
        ix = regrouped[start:start + CFG["BATCH_SIZE"]]
        live, length = _capture_parity_batch(ix, use_cache=True)
        same_layout = {key: np.stack([current_by_row[int(r)][key] for r in ix]) for key in live}
        sensitivity += _parity_rows("fit_regrouping_sensitivity", ix, live, same_layout, length, False)
        for method in ["e14_rule", "prototype__budget", "direction__budget", "vector_lr__budget"]:
            before, _, _ = choose_runtime(bundle, method, same_layout["HEAD_Q"],
                                           same_layout["HEAD_LAST"], same_layout["C_MASS"])
            after, _, _ = choose_runtime(bundle, method, live["HEAD_Q"], live["HEAD_LAST"], live["C_MASS"])
            for row, a, b in zip(ix, before, after):
                route_checks.append({"row": int(row), "method": method,
                                     "same_layout_action": str(a), "regrouped_action": str(b),
                                     "changed": bool(a != b)})
            if method != "e14_rule":
                scores, _ = infer_bundle(bundle, live["HEAD_Q"], live["HEAD_LAST"], live["C_MASS"])
                name, mode = method.split("__")
                assert np.array_equal(after, budget_actions(bundle, name, scores[name], mode)), "readout 구현 불일치"
    sensitivity = pd.DataFrame(sensitivity)
    routes = pd.DataFrame(route_checks)
    write_csv(sensitivity, ROOT / "FIT_BATCH_SENSITIVITY.csv")
    write_csv(routes, ROOT / "FIT_BATCH_ROUTE_SENSITIVITY.csv")
    display(routes.groupby("method").agg(n=("changed", "size"), changed=("changed", "sum")))
    if not sensitivity["within_original_tolerance"].all() or routes.changed.any():
        warnings.warn("같은 fit 입력도 batch 재구성 시 수치/경계 action이 달라졌습니다. "
                      "FIT_BATCH_* 파일에 기록했습니다. 원래 bank-layout 검사는 통과했으며, "
                      "QA의 기존 batching/readout/threshold는 변경하지 않습니다.")
    approval = {"patch": E17_PARITY_PATCH, "run_hash": RUN_HASH, "revision": REVISION,
                "model_object_id": id(MODEL), "same_layout_and_cache_checks_passed": True,
                "bank_batch_size": E14_BANK_BATCH_SIZE,
                "fit_rows": rows.tolist(), "thresholds_unchanged": True,
                "unlabeled_batch_companions_forward_only": sorted(
                    set(int(r) for ix in batches for r in ix) - target_set),
                "test_labels_or_gold_used": False,
                "bank_relative_limit": CFG["BANK_REL_ERROR_MAX"],
                "mass_absolute_limit": CFG["MASS_ABS_ERROR_MAX"],
                "torch": str(torch.__version__), "cuda": str(torch.version.cuda),
                "gpu": torch.cuda.get_device_name(0),
                "readout_lock_hashes": {str(f): b["lock_hash"] for f,b in FOLD_BUNDLES.items()},
                "fit_regrouped_action_changes": int(routes.changed.sum())}
    save_json(ROOT / "BANK_RUNTIME_PARITY_APPROVAL.json", approval)
    print("Bank/runtime 동일-layout 검사 PASS. 기존 5-fold lock을 재사용해 13번부터 이어갑니다.")
    return parity, approval


PARITY = []
if CFG["RUN_GENERATION"]:
    PARITY, BANK_PARITY_APPROVAL = _run_e14_parity()
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("RUN_GENERATION=False: GPU parity와 QA를 실행하지 않습니다.")


## 13–14. 고정한 모든 QA 조건 실행 및 평가

라우터 함수에는 row ID, K/C label, gold를 전달하지 않고 실제 activation만 전달합니다.
Generation을 마친 뒤 별도 단계에서 gold와 EM을 비교합니다.
중간 batch는 원자적으로 저장하며 run/readout/model/vector fingerprint가 다르면 재사용하지 않습니다.

**주 QA 비교:** `prototype__budget` vs fresh `e14_rule`, 그리고 둘 각각 vs fresh `noop`.
`prototype__state_map`은 no-op 과다 선택이 주요 제약인지 보기 위한 별도 조건입니다.
`prototype__budget_random_steer`는 route는 그대로 두고 steering 방향만 무작위화합니다.

출력은 Recovery, Damage, Net Gain, EM, 개입률, 상태별 ΔEM, 질문 단위 CI를 포함합니다.
은행(bank)에서 계산한 action과 live action 차이가 있으면 별도로 기록합니다.
실제 QA 논의에는 **live routing metric**을 우선 사용합니다.

In [ ]:
# 13. 모든 사전 지정 조건의 5-fold 실제 generation. test 결과로 조건을 고르지 않음.
QA_OUTPUTS=[]
if CFG["RUN_GENERATION"]:
    # 수정된 12번 검사를 통과한 같은 모델/lock에서만 QA를 시작합니다.
    _approval = globals().get("BANK_PARITY_APPROVAL")
    if not isinstance(_approval, dict) or not _approval.get("same_layout_and_cache_checks_passed"):
        raise RuntimeError("먼저 수정된 12번 bank/runtime 검사를 실행해 통과하세요. 실패한 검사를 건너뛰면 안 됩니다.")
    if (_approval.get("run_hash") != RUN_HASH or _approval.get("revision") != REVISION
            or _approval.get("model_object_id") != id(MODEL)):
        raise RuntimeError("모델/실행 설정이 parity 검사 이후 바뀌었습니다. 12번부터 다시 실행하세요.")
    if (_approval.get("bank_relative_limit") != CFG["BANK_REL_ERROR_MAX"]
            or _approval.get("mass_absolute_limit") != CFG["MASS_ABS_ERROR_MAX"]):
        raise RuntimeError("parity 검사 한도가 변경됐습니다. 12번부터 다시 실행하세요.")
    for _fold, _bundle in FOLD_BUNDLES.items():
        if (_approval["readout_lock_hashes"].get(str(_fold)) != _bundle["lock_hash"]
                or sha_file(_bundle["lock_path"]) != _bundle["lock_hash"]):
            raise RuntimeError("readout lock이 parity 검사 이후 변경됐습니다. QA를 중단합니다.")
    for fold,bundle in FOLD_BUNDLES.items():
        for method in QA_METHODS:
            print("\n=== fold",fold,"|",method,"===",flush=True)
            out=cached_generate(bundle["test_rows"],bundle,method)
            # 답변 생성이 끝난 후에만 gold로 채점합니다.
            out["em"]=[int(norm_answer(truncate_answer(raw)) in {norm_answer(g) for g in FRAME.iloc[int(r)].aliases})
                       for r,raw in zip(out.row,out.raw)]
            out["true_state"]=FRAME.iloc[out.row.to_numpy()].state.to_numpy()
            out["question"]=FRAME.iloc[out.row.to_numpy()].question.to_numpy()
            write_csv(out,ROOT/f"fold_{fold}"/f"QA_SCORED_{method}.csv")
            QA_OUTPUTS.append(out)
            gc.collect(); torch.cuda.empty_cache()
    QA_OOF=pd.concat(QA_OUTPUTS,ignore_index=True).sort_values(["method","row"])
    assert QA_OOF.groupby("method").size().eq(1224).all()
    assert QA_OOF.groupby("method").row.nunique().eq(1224).all()
    write_csv(QA_OOF,ROOT/"QA_OOF_PREDICTIONS.csv")
    if "prototype__budget_random_steer" in QA_METHODS:
        a=QA_OOF[QA_OOF.method.eq("prototype__budget")].sort_values("row")
        b=QA_OOF[QA_OOF.method.eq("prototype__budget_random_steer")].sort_values("row")
        assert np.array_equal(a.action,b.action), "random steering control에서 route가 바뀌었습니다."
    print("All requested QA generations completed.")
else:
    QA_OOF=pd.DataFrame()
    print("RUN_GENERATION=False: QA 측정값은 만들지 않습니다. 4-state/routing 점수만 저장되었습니다.")


In [ ]:
# 14. QA 결과: baseline 대비 Recovery/Damage/Net Gain, 질문별 bootstrap, fold별 결과

def bootstrap_mean_by_question(delta,questions,n_boot=2000,seed=42):
    d=pd.DataFrame({"q":questions,"d":np.asarray(delta,float)}).groupby("q")["d"].mean().to_numpy()
    rng=np.random.default_rng(seed); values=[]
    for start in range(0,n_boot,200):
        ix=rng.integers(len(d),size=(min(200,n_boot-start),len(d)))
        values.extend(d[ix].mean(1).tolist())
    lo,hi=np.quantile(values,[.025,.975])
    return float(np.mean(d)),float(lo),float(hi)

def qa_compare(base,active):
    a=active.sort_values("row"); b=base.sort_values("row")
    assert np.array_equal(a.row,b.row)
    delta=a.em.to_numpy(int)-b.em.to_numpy(int)
    point,lo,hi=bootstrap_mean_by_question(delta,a.question.to_numpy(),CFG["N_BOOT"],CFG["SEED"])
    result={"n":len(a),"baseline_EM":b.em.mean(),"steered_EM":a.em.mean(),
            "Recovery":int((delta==1).sum()),"Damage":int((delta==-1).sum()),"NetGain":int(delta.sum()),
            "DeltaEM":point,"DeltaEM_CI_low":lo,"DeltaEM_CI_high":hi,
            "intervention_rate":a.action.ne("noop").mean(),
            "seconds_per_row_batch_average":a.seconds_per_row_batch_average.mean(),
            "route_ms_batch_mean":a.route_ms_batch.mean()}
    for st,nm in enumerate(STATE_NAMES):
        ix=a.true_state.eq(st).to_numpy()
        result[nm+"_DeltaEM"]=float(delta[ix].mean()); result[nm+"_n"]=int(ix.sum())
    return result

QA_SUMMARY=pd.DataFrame(); QA_PAIRWISE=pd.DataFrame(); QA_RUNTIME_ACTION=pd.DataFrame()
if len(QA_OOF):
    baseline=QA_OOF[QA_OOF.method.eq("noop")]
    records=[]; fold_records=[]; runtime_action_records=[]
    for method,g in QA_OOF.groupby("method"):
        records.append({"method":method,**qa_compare(baseline,g)})
        for fold,fg in g.groupby("fold"):
            bg=baseline[baseline.fold.eq(fold)]
            fold_records.append({"fold":fold,"method":method,**qa_compare(bg,fg)})
        am,_,_,_,_=classification_metrics(state_action(g.true_state),g.action,ACTION_NAMES)
        runtime_action_records.append({"method":method,**am})
    QA_SUMMARY=pd.DataFrame(records); QA_FOLDS=pd.DataFrame(fold_records)
    QA_RUNTIME_ACTION=pd.DataFrame(runtime_action_records)
    live_states=[]; replay=[]
    for method,g in QA_OOF.groupby("method"):
        g=g.sort_values("row")
        if g.pred_state.notna().all():
            sm,_,_,_,_=classification_metrics(g.true_state,g.pred_state.astype(int),list(range(4)))
            live_states.append({"method":method,**sm})
        offline_name=method.removesuffix("_random_steer")
        old=ACTION_OOF[ACTION_OOF.method.eq(offline_name)].sort_values("row")
        if len(old)==len(g):
            assert np.array_equal(old.row,g.row)
            replay.append({"method":method,"n":len(g),
                           "live_vs_bank_action_mismatch":int((old.action.to_numpy()!=g.action.to_numpy()).sum())})
    QA_LIVE_STATE=pd.DataFrame(live_states); QA_REPLAY=pd.DataFrame(replay)
    write_csv(QA_LIVE_STATE,ROOT/"QA_LIVE_STATE_METRICS.csv")
    write_csv(QA_REPLAY,ROOT/"QA_LIVE_VS_BANK_ACTION_AUDIT.csv")
    if len(QA_REPLAY) and QA_REPLAY.live_vs_bank_action_mismatch.sum():
        warnings.warn("bank 계산과 live routing의 차이가 있습니다. 실제 QA의 routing 평가는 QA_LIVE_ROUTING_METRICS를 보고하세요.")
    pairwise=[]
    for method in ["prototype__budget","direction__budget","prototype__state_map","vector_lr__budget"]:
        if method not in QA_METHODS: continue
        a=QA_OOF[QA_OOF.method.eq(method)].sort_values("row")
        e14=QA_OOF[QA_OOF.method.eq("e14_rule")].sort_values("row")
        d,lo,hi=bootstrap_mean_by_question(a.em.to_numpy()-e14.em.to_numpy(),a.question.to_numpy(),CFG["N_BOOT"],CFG["SEED"]+1)
        pairwise.append({"method":method,"reference":"e14_rule","DeltaEM":d,"CI_low":lo,"CI_high":hi})
    QA_PAIRWISE=pd.DataFrame(pairwise)
    for df,name in [(QA_SUMMARY,"QA_OOF_SUMMARY"),(QA_FOLDS,"QA_FOLD_METRICS"),
                    (QA_PAIRWISE,"QA_PAIRED_VS_E14"),(QA_RUNTIME_ACTION,"QA_LIVE_ROUTING_METRICS")]:
        write_csv(df,ROOT/(name+".csv"))
    display(QA_SUMMARY[["method","n","baseline_EM","steered_EM","Recovery","Damage","NetGain",
                        "DeltaEM","DeltaEM_CI_low","DeltaEM_CI_high","intervention_rate"]])
    display(QA_PAIRWISE)
    # 기존 E14 출력은 별도 참고. 자동으로 답변을 대체하거나 재채점 기준을 바꾸지 않습니다.
    old=E14_ROOT/"OOF_HEAD_LABELFREE_GATE_SUMMARY.json"
    if old.exists():
        save_json(ROOT/"E14_HISTORICAL_QA_REFERENCE.json",read_json(old))
        print("기존 E14 QA는 별도 reference로 저장. E17 공통 engine의 fresh baseline과 동일하다고 가정하지 않습니다.")
else:
    print("QA 미실행: Net Gain/EM 결과 없음.")


## 15. 보고서·공유 요약 자동 저장

**핵심 파일**
- `STATE_OOF_SUMMARY.csv`: 같은 head 집합에서 각 readout의 4-state F1
- `ACTION_OOF_SUMMARY.csv`: K/C-derived action target 진단
- `PAIRED_STATE_F1_BOOTSTRAP.csv`: paired F1 차이와 질문 bootstrap CI
- `QA_OOF_SUMMARY.csv`: 실제 답변 EM / Recovery / Damage / Net Gain
- `QA_PAIRED_VS_E14.csv`: fresh E14 gate 대비 QA 차이
- `QA_LIVE_STATE_METRICS.csv`, `QA_LIVE_VS_BANK_ACTION_AUDIT.csv`: runtime 검증
- `E17_RESULT_REPORT.md`, `E17_RESULTS_SHARE.zip`: 보고용 요약

**해석 경계**
- `perhead_scalar_lr > scalar_lr`: 3-score 집계 단계가 한계일 가능성.
- `vector_lr > perhead_scalar_lr`: 같은 head의 vector 좌표/방향이 유용할 가능성.
- `prototype/direction > e14_median4`: 고정된 기하학 규칙의 개선 가능성.
- `geometry < vector_lr`: 현재 geometry가 학습된 decision boundary를 충분히 대체하지 못함.
- state F1 상승·QA 정체: state→action 또는 actuator 효과도 별도로 점검해야 함.

모든 성능은 실행 후 채워지며, 예상 수치를 실제 결과로 넣지 않았습니다.

In [ ]:
# 15. 교수님 보고용 Markdown/그림 + 결과 ZIP (토큰/원본 입력/모델 가중치 포함 안 함)
import matplotlib.pyplot as plt
import zipfile

view=STATE_SUMMARY[STATE_SUMMARY.method.isin(PRIMARY_NAMES)].copy()
order=["e14_median4","scalar_lr","perhead_scalar_lr","direction","prototype","pca_prototype","vector_lr"]
view=view.set_index("method").reindex(order)
fig,ax=plt.subplots(figsize=(9,4.2))
ax.bar(view.index,view.OOF_macro_F1)
ax.set_ylabel("4-state Macro-F1 (pooled OOF)"); ax.set_ylim(0,1)
ax.set_title("E17: matched head union, different readouts")
ax.tick_params(axis="x",rotation=20); fig.tight_layout()
fig.savefig(ROOT/"E17_STATE_READOUTS.png",dpi=160); plt.show(); plt.close(fig)

pr=STATE_SUMMARY.set_index("method")
lines=["# E17 결과 요약", "", "## [문제상황]",
       "E14에서 관찰된 vector-decoding과 scalar-rule 성능 차이가 readout 설계와 관련되는지 동일 head에서 비교했다.",
       "", "## [해결책]", "동일 question-wise 5-fold, fold별 E14 role-head union, 같은 neutral bank를 사용했다.",
       "Vector feature = HQ + (HL-HQ) + 동일 head C_MASS. scalar = memory/context/conflict 3점수. Per-head scalar LR는 head별 norm/shift/mass를 그대로 사용한다.",
       "방향/프로토타입은 fit K/C 라벨을 사용한 geometric classifier이며, LR 최적화만 수행하지 않는다.",
       "별도 learned classifier 전체가 없는 방법 또는 training-free/label-free 방법이라고 주장하지 않는다.",
       "", "## [효과: 측정값]", view.reset_index().to_markdown(index=False), "",
       "### Paired F1 comparison", PAIRED_STATE.to_markdown(index=False), "",
       "### Action target 진단", "target = K+C- retain, K-C+ accept, 나머지 noop. QA 최적 action 정답이 아님.",
       ACTION_SUMMARY[ACTION_SUMMARY.method.isin(["e14_rule","all_noop","direction__budget","prototype__budget", "prototype__state_map","vector_lr__budget","scalar_lr__budget"])].to_markdown(index=False),
       "", "### Controls", CONTROL_SUMMARY.to_markdown(index=False), ""]
if len(QA_SUMMARY):
    lines += ["## 실제 QA", QA_SUMMARY.to_markdown(index=False), "", "### E14 대비 paired QA",QA_PAIRWISE.to_markdown(index=False)]
else:
    lines += ["## 실제 QA", "RUN_GENERATION=False로 QA를 실행하지 않았음. state F1 개선을 QA 개선으로 대체해서 보고하면 안 됨."]
lines += ["", "## 해석 시 제한",
          "1. geometric readout도 supervised classifier이다. LR/MLP optimizer-free라는 뜻과 구분한다.",
          "2. E14/기존 p8s 결과를 보고 설계한 후속 분석이며 독립적인 새 blind test가 아니다.",
          "3. 동일 fold의 head lock만 사용했다. 다른 outer-fold discovery artifact를 합치지 않았다.",
          "4. 선택된 head를 고정한 조건부 비교다. 전체 discovery 과정의 label shuffle 검정은 아니다.",
          "5. bootstrap은 질문 단위의 고정 예측 재표집이며 전체 학습/선택 불확실성을 포함하지 않는다.",
          "6. raw AUROC는 fold별로 계산 후 평균 보고했다. fold 간 score scale이 달라 raw pooled AUROC는 주 지표로 쓰지 않는다.",
          "7. E14의 기존 0.6306은 다른 head-set/feature의 supervised 분석 값이다. 이 표에 대입하지 않았다.",
          "8. state 판별 개선, action-target 일치도, 실제 QA 이득은 서로 다른 주장이다.",
          "9. budget은 policy_cal에서 E14와 비슷한 개입률이 되도록 정한 비교이며 test 개입률의 완전 일치를 보장하지 않는다.",
          "10. direction/prototype 역시 최종적으로 소수 score로 투영된다. 벡터의 모든 정보를 보존했다고 주장하지 않는다.",
          "11. 모든 조건에 같은 capture hook을 넣었으므로 end-to-end timing은 instrumentation을 포함한다. LR 제거만으로 latency 이득을 보장하지 않는다.",
          "", "## 조건별 읽는 법",
          "- vector_lr > perhead_scalar_lr: 같은 head별 크기/shift/mass보다 좌표 방향 정보가 유용할 가능성을 지지.",
          "- perhead_scalar_lr > scalar_lr: head별 scalar를 3개 aggregate로 합치는 단계의 손실 가능성을 지지.",
          "- vector_lr > scalar_lr: 같은 LR 계열에서 전체 vector feature가 더 유용한지 확인. 최적 achievable 성능의 upper bound는 아님.",
          "- scalar_lr > E14 median: 기존 threshold/결합 규칙도 병목일 수 있음.",
          "- geometry > E14 median, geometry ≈ vector_lr: 단순 고정 geometry로 정보 활용 가능성을 지지.",
          "- geometry < vector_lr: 학습된 discriminative boundary가 중요할 수 있음.",
          "- state F1 상승, QA 정체: state-to-action mapping 또는 actuator 이득을 별도로 점검해야 함."]
report="\n\n".join(lines)
atomic_text(ROOT/"E17_RESULT_REPORT.md",report)
zip_path=ROOT/"E17_RESULTS_SHARE.zip"
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob("*"):
        if not p.is_file() or p==zip_path: continue
        rel=p.relative_to(ROOT)
        # 원문 질문/답변/세부 row predictions와 큰 readout 가중치는 공유 묶음에서 제외.
        if "generation" in rel.parts or "PREDICTIONS" in p.name or p.name=="READOUT_LOCK.json" or p.name.startswith("QA_SCORED_"): continue
        if p.suffix in {".csv",".json",".md",".png"}: z.write(p,str(rel))
print("보고서:",ROOT/"E17_RESULT_REPORT.md")
print("요약 공유 ZIP:",zip_path)
print("원본 E14/E15/E16 파일은 수정하지 않았습니다.")


## 참고: 기존 E15와 E17의 차이

E15는 `L13 full hidden + 기존 probe 출력 → action별 예상 ΔEM → 방향·강도 선택`입니다.
저장된 Fold-0 test 246행 결과는 baseline EM 0.5244, online utility EM 0.5569,
Recovery 12 / Damage 4 / Net Gain +8입니다. 별도의 learned utility router를 사용하는 실험입니다.

E17은 **readout 통제 비교**입니다. 강도/action space를 새로 검색하지 않고 E14 actuator를 유지합니다.
E15의 최종 test 결과는 outer 5-fold가 아니라 Fold 0이며, 내부 5-fold는 개발용 utility OOF입니다.
E15의 결과를 E17의 5-fold 결과표에 섞지 않습니다.

## 공식 참고자료 및 출처

- E14 원본: https://colab.research.google.com/drive/1D68pg1E_N3NJBZinjNB2XiyKA62ZuPM8
- E15 원본: https://colab.research.google.com/drive/15Za8w_bDe5t6mduP_5SevfsWKCQSRKZS
- scikit-learn Nearest Centroid (centroid도 classifier): https://scikit-learn.org/1.5/modules/neighbors.html#nearest-centroid-classifier
- scikit-learn leakage / feature-selection 분리 원칙: https://scikit-learn.org/stable/common_pitfalls.html#data-leakage
- Hugging Face cache / cache_position: https://huggingface.co/docs/transformers/v4.57.0/en/cache_explanation

## 전달 전 검사 범위

코드 구문, 합성 activation 기반 5-fold 분석/출력, metric 계산, freeze/resume,
평가 label 변경에 대한 inference 불변성, PyTorch 모의 모델의 hook/cap/cache를 검사했습니다.
이 검사는 **실제 E14 activation 성능 측정이나 Llama A100 실행이 아닙니다**.
작성 환경의 sklearn은 1.8.0, 실제 Colab 설정은 기존과 같은 1.6.1입니다.
최종 notebook의 GPU bank-parity 검사를 통과한 뒤 실제 실험 결과를 해석해야 합니다.

## 16–18. [추가 실험] Direction이 retain/accept로 판단한 입력은 전부 steering

**추가 조건: `direction__state_map`** — `e17-direction-state-map-v1`

| Direction의 예측 상태 | 실제 action |
|---|---|
| K+C− | **retain — 추가 budget/확신도 조건 없이 적용** |
| K−C+ | **accept — 추가 budget/확신도 조건 없이 적용** |
| K+C+ / K−C− | no-op |

`direction__budget`를 삭제하거나 결과를 덮어쓰지 않습니다.
**기존 head, Direction 판정 방향, 5-fold split, steering vector·강도·레이어는 그대로**입니다.
전체 입력 100%에 강제 개입하는 것이 아니라 **retain/accept로 판정된 후보 100%에 개입**합니다.
판정 자체의 K/C 경계는 유지하고, 그 뒤의 추가 margin/개입률 제한만 사용하지 않습니다.
**이 조건도 conditional steering입니다.** 입력의 K/C 예측에 따라 retain/accept/no-op이 달라집니다.
`budget` 제거는 조건부 개입 원칙의 폐기가 아니라, 두 번째 보수적 제한이 필요한지 보는 ablation입니다.
오분류한 입력에도 개입할 수 있으므로 Recovery와 함께 Damage를 반드시 평가합니다.

### 실행
기존 E17 런타임이 살아 있으면 **16 → 17 → 18번만 순서대로 실행**하세요.
새 런타임이면 **1~12번으로 데이터·고정 readout·모델·parity를 복구한 뒤 16번으로 이동**하세요.
Drive에 기존 8조건의 생성 cache가 완료돼 있어야 합니다. 13~15번 기존 QA는 다시 실행할 필요 없습니다.

17번은 새 조건 하나에 대해서만 5-fold **실제 generation**을 수행합니다(총 1,224행).
완료된 새 조건의 batch는 중단 후 재개할 때 재사용합니다. 이전 비교군은 cache를 검증해 **읽기만** 합니다.
18번에서 기존 `direction__budget`, `e14_rule`, `prototype__state_map`과 실제 EM/Recovery/Damage/Net Gain을 비교합니다.
**실제 생성 중 routing F1과 bank/live 차이도 별도로 출력**합니다.

결과 위치: `E17_geometry_matched_heads_v1/extensions/direction_state_map_v1/`.
기존 결과를 본 뒤 요청한 탐색적 후속 비교이며, 새로운 blind test 또는 사전 지정 주 결과로 표기하지 않습니다.


In [ ]:
# 16. Direction: 판정된 retain/accept 후보 전부 개입 — 추가 budget/margin 제한 없음
# ADDON: e17-direction-state-map-v1. 기존 CFG/RUN_HASH/READOUT_LOCK/8조건 결과는 변경하지 않습니다.
DSM_VERSION = "e17-direction-state-map-v1"
DSM_METHOD = "direction__state_map"
DSM_REFERENCES = ["noop", "e14_rule", "direction__budget", "prototype__state_map"]
_needed = ["CFG", "ROOT", "RUN_HASH", "FOLD_BUNDLES", "FRAME", "HQ", "HL", "CM",
           "MODEL", "TOK", "REVISION", "torch", "choose_runtime", "cached_generate",
           "state_action", "action_margin", "apply_margin", "vector_features", "role_scores",
           "read_json", "save_json", "write_csv", "sha_file", "object_hash",
           "classification_metrics", "BANK_PARITY_APPROVAL", "VECTOR_PATHS",
           "np", "pd", "gc", "STATE_NAMES", "ACTION_NAMES", "atomic_text",
           "norm_answer", "truncate_answer", "display"]
_missing = [key for key in _needed if key not in globals()]
if _missing or globals().get("MODEL") is None:
    raise RuntimeError(
        "기존 E17 런타임 준비가 필요합니다. 1~12번을 순서대로 실행한 뒤 16번으로 오세요. "
        "저장된 READOUT_LOCK을 재사용하므로 재학습하지 않습니다. "
        "이미 완료한 13~15번 QA는 다시 실행할 필요 없습니다. 누락: " + ", ".join(_missing))
if not CFG["RUN_GENERATION"] or not torch.cuda.is_available():
    raise RuntimeError("이번 추가 조건은 실제 QA 생성입니다. RUN_GENERATION=True와 GPU가 필요합니다.")
if sorted(FOLD_BUNDLES) != [0, 1, 2, 3, 4]:
    raise RuntimeError("5개 outer fold가 모두 준비되어야 합니다.")


def _dsm_check_frozen_runtime():
    manifest = read_json(ROOT / "RUN_MANIFEST.json")
    current = {k: v for k, v in CFG.items() if k not in {"RUN_GENERATION", "RESUME"}}
    if object_hash(manifest) != RUN_HASH or object_hash(current) != object_hash(manifest["config"]):
        raise RuntimeError("기존 E17 설정이 바뀌었습니다. 개입 정책 외의 조건은 원래대로 유지하세요.")
    approval = BANK_PARITY_APPROVAL
    if (not isinstance(approval, dict)
            or not approval.get("same_layout_and_cache_checks_passed")
            or approval.get("run_hash") != RUN_HASH
            or approval.get("revision") != REVISION
            or approval.get("model_object_id") != id(MODEL)):
        raise RuntimeError("같은 모델/설정으로 12번 parity 검사를 먼저 통과해야 합니다.")
    if (approval.get("bank_relative_limit") != CFG["BANK_REL_ERROR_MAX"]
            or approval.get("mass_absolute_limit") != CFG["MASS_ABS_ERROR_MAX"]):
        raise RuntimeError("parity 검사 기준이 바뀌었습니다. 원래 기준으로 12번을 실행하세요.")
    for fold, bundle in FOLD_BUNDLES.items():
        if (sha_file(bundle["lock_path"]) != bundle["lock_hash"]
                or approval["readout_lock_hashes"].get(str(fold)) != bundle["lock_hash"]):
            raise RuntimeError(f"fold {fold}: readout lock이 변경됐습니다.")
        saved = read_json(bundle["lock_path"])
        if (object_hash(bundle["models"]["direction"].as_dict()) != object_hash(saved["models"]["direction"])
                or object_hash(bundle["policy"]) != object_hash(saved["policy"])
                or list(bundle["heads"]) != list(saved["heads"])
                or object_hash(bundle["margins"]) != object_hash(saved["margins"])
                or list(map(int, bundle["test_rows"])) != list(saved["test_rows"])):
            raise RuntimeError(f"fold {fold}: 메모리의 판정 설정이 저장된 lock과 다릅니다.")
        if bundle["margins"]["direction"]["state_map"] != 0.0:
            raise RuntimeError("state_map은 추가 margin 제한이 없는 cutoff=0이어야 합니다.")


def _dsm_read_cached_only(bundle, method):
    """기존 비교군은 생성하지 않고 동일 fingerprint의 완료된 batch만 읽습니다."""
    rows = list(map(int, bundle["test_rows"]))
    folder = ROOT / f"fold_{bundle['fold']}" / "generation" / method
    fingerprint = object_hash({"run": RUN_HASH, "readout_lock": bundle["lock_hash"],
                               "method": method, "revision": REVISION,
                               "vectors": sha_file(VECTOR_PATHS[bundle["fold"]]), "rows": rows})
    expected = {"fingerprint": fingerprint, "rows": rows, "method": method}
    header = folder / "CACHE_MANIFEST.json"
    if not header.exists() or read_json(header) != expected:
        raise RuntimeError(f"기존 비교군 cache가 없거나 조건이 다릅니다: {header}. 기존 13번 완료 여부를 확인하세요.")
    records = []
    for start in range(0, len(rows), CFG["BATCH_SIZE"]):
        ix = rows[start:start + CFG["BATCH_SIZE"]]
        path = folder / f"batch_{start:06d}.json"
        if not path.exists():
            raise RuntimeError(f"기존 비교군이 아직 완료되지 않았습니다: {path}")
        cache = read_json(path)
        if cache["fingerprint"] != fingerprint or cache["rows"] != ix:
            raise RuntimeError(f"기존 cache fingerprint/rows 불일치: {path}")
        rec = cache["records"]
        if ([r["row"] for r in rec] != ix
                or any(r["method"] != method or int(r["fold"]) != bundle["fold"] for r in rec)):
            raise RuntimeError(f"기존 cache record 불일치: {path}")
        records.extend(rec)
    return pd.DataFrame(records).sort_values("row").reset_index(drop=True)


def _dsm_score_completed(raw):
    """생성/라우팅을 끝낸 결과에만 gold와 K/C 라벨을 결합합니다."""
    out = raw.copy()
    rows = out.row.to_numpy(int)
    out["em"] = [int(norm_answer(truncate_answer(text)) in
                     {norm_answer(g) for g in FRAME.iloc[int(row)].aliases})
                 for row, text in zip(rows, out.raw)]
    out["true_state"] = FRAME.iloc[rows].state.to_numpy(int)
    out["question"] = FRAME.iloc[rows].question.to_numpy()
    return out


_dsm_check_frozen_runtime()
# 가장 높은 state score가 retain/accept이면 margin 크기와 무관하게 실행되는지 검증.
# exact tie는 기존 argmax tie-break를 그대로 유지합니다. 정답 라벨은 사용하지 않습니다.
_test_scores = np.array([[4., 0., 0., 0.], [0., 4., 0., 0.], [0., 0., 4., 0.],
                         [0., 0., 0., 4.], [0., 1e-12, 0., 0.], [0., 0., 0., 1e-12],
                         [0., 2., 2., 0.], [0., 0., 2., 2.], [0., 0., 0., 0.]])
np.testing.assert_array_equal(apply_margin(_test_scores, 0.), state_action(_test_scores.argmax(1)))

# 모델을 다시 실행하기 전에 기존 비교군 cache가 완성되었는지 확인합니다.
DSM_REFERENCE_RAW = {}
for _method in DSM_REFERENCES:
    DSM_REFERENCE_RAW[_method] = pd.concat(
        [_dsm_read_cached_only(FOLD_BUNDLES[f], _method) for f in range(5)], ignore_index=True
    ).sort_values("row").reset_index(drop=True)
    assert DSM_REFERENCE_RAW[_method].row.tolist() == list(range(len(FRAME)))

DSM_ROOT = ROOT / "extensions" / "direction_state_map_v1"
DSM_MANIFEST = {
    "extension": DSM_VERSION, "parent_run_hash": RUN_HASH,
    "method": DSM_METHOD, "references": DSM_REFERENCES, "revision": REVISION,
    "readout_lock_hashes": {str(f): b["lock_hash"] for f, b in FOLD_BUNDLES.items()},
    "rule": {"pred_K+C+": "noop", "pred_K+C-": "retain", "pred_K-C-": "noop", "pred_K-C+": "accept"},
    "extra_budget_or_margin_gate": False, "test_labels_for_routing": False,
    "retrain_readout": False, "steering_strength_and_layers_changed": False,
    "scope": "user-requested exploratory follow-up after original E17 results; not a new blind test",
    "reference_cache_hashes": {
        f"{f}/{m}": sha_file(ROOT / f"fold_{f}" / "generation" / m / "CACHE_MANIFEST.json")
        for f in range(5) for m in DSM_REFERENCES},
}
DSM_ROOT.mkdir(parents=True, exist_ok=True)
_dsm_manifest_path = DSM_ROOT / "EXTENSION_MANIFEST.json"
if _dsm_manifest_path.exists() and read_json(_dsm_manifest_path) != DSM_MANIFEST:
    raise RuntimeError("동일 추가실험 ID에 다른 설정이 저장되어 있습니다. 기존 결과를 덮어쓰지 않습니다.")
if not _dsm_manifest_path.exists():
    # 이름이 같은 출처 불명의 이전 cache를 이번 추가실험으로 둔갑시키지 않습니다.
    unknown = [str(ROOT / f"fold_{f}" / "generation" / DSM_METHOD / "CACHE_MANIFEST.json")
               for f in range(5)
               if (ROOT / f"fold_{f}" / "generation" / DSM_METHOD / "CACHE_MANIFEST.json").exists()]
    if unknown:
        raise RuntimeError("이름이 같은 cache가 이미 있으나 추가실험 manifest가 없습니다. 출처 확인 필요: " + str(unknown))
    save_json(_dsm_manifest_path, DSM_MANIFEST)

# CPU bank 진단: 각 test 입력에서 추가 제한이 실제로 0개인지 확인. cutoff 재탐색 없음.
_dsm_bank_rows, _dsm_checks = [], []
for _fold, _bundle in FOLD_BUNDLES.items():
    _rows = np.asarray(_bundle["test_rows"], int)
    _actions, _pred, _margin = choose_runtime(_bundle, DSM_METHOD, HQ[_rows], HL[_rows], CM[_rows])
    _expected = state_action(_pred)
    np.testing.assert_array_equal(_actions, _expected)
    _budget, _budget_pred, _ = choose_runtime(_bundle, "direction__budget", HQ[_rows], HL[_rows], CM[_rows])
    np.testing.assert_array_equal(_budget_pred, _pred)
    _dsm_checks.append({"fold": _fold, "n": len(_rows),
                        "predicted_retain_or_accept": int((_expected != "noop").sum()),
                        "state_map_interventions": int((_actions != "noop").sum()),
                        "suppressed_after_state_decision": int((_actions != _expected).sum()),
                        "budget_interventions": int((_budget != "noop").sum())})
    _dsm_bank_rows.append(pd.DataFrame({"row": _rows, "fold": _fold,
                                      "pred_state": _pred, "action": _actions}))
DSM_BANK = pd.concat(_dsm_bank_rows, ignore_index=True).sort_values("row").reset_index(drop=True)
DSM_CHECKS = pd.DataFrame(_dsm_checks)
write_csv(DSM_CHECKS, DSM_ROOT / "ALL_ELIGIBLE_ROUTING_AUDIT.csv")
write_csv(DSM_BANK, DSM_ROOT / "BANK_DIRECTION_STATE_MAP_PREDICTIONS.csv")
display(DSM_CHECKS)
print("PASS: K+C- 예측은 전부 retain, K-C+ 예측은 전부 accept. 나머지 두 상태만 noop.")
print("판정 방향/head/steering 강도는 유지. 기존 budget 결과 유지. 다음 17번에서 새 조건만 생성합니다.")


In [ ]:
# 17. direction__state_map만 실제 generation (기존 8개 조건 재생성하지 않음)
_dsm_check_frozen_runtime()
if read_json(DSM_ROOT / "EXTENSION_MANIFEST.json") != DSM_MANIFEST:
    raise RuntimeError("추가실험 설정이 16번 이후 변경되었습니다.")
DSM_OUTPUTS = []
for _fold, _bundle in FOLD_BUNDLES.items():
    print(f"\n=== fold {_fold} | {DSM_METHOD} | 추가 budget 제한 없음 ===", flush=True)
    _new = cached_generate(_bundle["test_rows"], _bundle, DSM_METHOD)
    if _new.pred_state.isna().any():
        raise RuntimeError("runtime predicted state가 기록되지 않았습니다.")
    _expected = state_action(_new.pred_state.to_numpy(int))
    np.testing.assert_array_equal(_new.action.to_numpy(), _expected,
                                  err_msg="판정 후 action이 억제/변경되었습니다.")
    _ratios = _new.max_delta_ratio.to_numpy(float)
    _active = _expected != "noop"
    if (not np.isfinite(_ratios).all() or np.any(_ratios[_active] <= 0.)
            or np.any(_ratios[~_active] != 0.)
            or np.any(_ratios > CFG["MAX_DELTA_RATIO"] * 1.025 + 1e-5)):
        raise RuntimeError("개입 대상으로 판정한 행의 실제 steering 적용/강도 검사가 실패했습니다.")
    _new = _dsm_score_completed(_new)
    write_csv(_new, DSM_ROOT / f"fold_{_fold}" / "QA_DIRECTION_STATE_MAP_PREDICTIONS.csv")
    DSM_OUTPUTS.append(_new)
    gc.collect()
    torch.cuda.empty_cache()
DSM_OOF = pd.concat(DSM_OUTPUTS, ignore_index=True).sort_values("row").reset_index(drop=True)
assert DSM_OOF.row.tolist() == list(range(len(FRAME)))
write_csv(DSM_OOF, DSM_ROOT / "QA_DIRECTION_STATE_MAP_OOF_PREDICTIONS.csv")
_dsm_check_frozen_runtime()
print("완료: direction__state_map 5-fold 실제 생성. 모든 retain/accept 판정에 steering 적용을 확인했습니다.")
print("기존 8개 조건의 결과는 덮어쓰지 않았습니다. 18번에서 비교합니다.")


In [ ]:
# 18. 기존 budget 조건 vs 판정된 후보 전부 개입: 실제 QA/live routing 비교
# 기존 요약 CSV를 덮어쓰지 않고 extensions/direction_state_map_v1에 별도 저장.

def _dsm_bootstrap(delta, questions):
    d = pd.DataFrame({"q": questions, "d": np.asarray(delta, float)}).groupby("q")["d"].mean().to_numpy()
    rng = np.random.default_rng(CFG["SEED"])
    values = []
    for start in range(0, CFG["N_BOOT"], 200):
        ix = rng.integers(len(d), size=(min(200, CFG["N_BOOT"] - start), len(d)))
        values.extend(d[ix].mean(1).tolist())
    lo, hi = np.quantile(values, [.025, .975])
    return float(d.mean()), float(lo), float(hi)


def _dsm_qa_compare(base, active):
    b = base.sort_values("row"); a = active.sort_values("row")
    np.testing.assert_array_equal(a.row, b.row)
    delta = a.em.to_numpy(int) - b.em.to_numpy(int)
    point, lo, hi = _dsm_bootstrap(delta, a.question.to_numpy())
    out = {"n": len(a), "baseline_EM": float(b.em.mean()), "steered_EM": float(a.em.mean()),
           "Recovery": int((delta == 1).sum()), "Damage": int((delta == -1).sum()),
           "NetGain": int(delta.sum()), "DeltaEM": point,
           "CI_low": lo, "CI_high": hi, "intervention_rate": float(a.action.ne("noop").mean()),
           "retain_count": int(a.action.eq("retain").sum()), "accept_count": int(a.action.eq("accept").sum())}
    for st, name in enumerate(STATE_NAMES):
        mask = a.true_state.eq(st).to_numpy()
        out[name + "_DeltaEM"] = float(delta[mask].mean()) if mask.any() else None
    return out


# 통계는 모두 저장된 실제 생성 결과로 계산합니다. routing에는 gold를 전달하지 않습니다.
DSM_RESULTS = {m: _dsm_score_completed(r) for m, r in DSM_REFERENCE_RAW.items()}
DSM_RESULTS[DSM_METHOD] = DSM_OOF
_dsm_base = DSM_RESULTS["noop"]
_order = ["e14_rule", "direction__budget", DSM_METHOD, "prototype__state_map"]
DSM_QA_SUMMARY = pd.DataFrame([{"method": m, **_dsm_qa_compare(_dsm_base, DSM_RESULTS[m])} for m in _order])
DSM_QA_FOLDS = pd.DataFrame([
    {"fold": f, "method": m,
     **_dsm_qa_compare(_dsm_base[_dsm_base.fold.eq(f)], DSM_RESULTS[m][DSM_RESULTS[m].fold.eq(f)])}
    for f in range(5) for m in _order])
DSM_PAIRED = pd.DataFrame([
    {"method": DSM_METHOD, "reference": ref, **_dsm_qa_compare(DSM_RESULTS[ref], DSM_OOF)}
    for ref in ["direction__budget", "e14_rule", "prototype__state_map"]])

_live = []
for m in _order:
    g = DSM_RESULTS[m].sort_values("row")
    truth = state_action(g.true_state.to_numpy(int))
    metrics, cm, prec, rec, f1 = classification_metrics(truth, g.action.to_numpy(), ACTION_NAMES)
    row = {"method": m, **metrics}
    for i, name in enumerate(ACTION_NAMES):
        row[name + "_recall"] = float(rec[i]); row[name + "_precision"] = float(prec[i])
    if g.pred_state.notna().all():
        sm, _, _, _, _ = classification_metrics(g.true_state.to_numpy(int), g.pred_state.to_numpy(int), list(range(4)))
        row["live_4state_macro_F1"] = sm["macro_F1"]
    _live.append(row)
DSM_LIVE_ROUTING = pd.DataFrame(_live)

# bank/live 차이는 F1 및 QA와 별개로 기록. 동일한 offline 값을 runtime 값으로 대체하지 않습니다.
_dsm_bank = DSM_BANK.sort_values("row")
_dsm_live = DSM_OOF.sort_values("row")
np.testing.assert_array_equal(_dsm_bank.row, _dsm_live.row)
_dsm_budget_live = DSM_RESULTS["direction__budget"].sort_values("row")
np.testing.assert_array_equal(_dsm_budget_live.row, _dsm_live.row)
DSM_LIVE_AUDIT = pd.DataFrame([{
    "n": len(DSM_OOF),
    "state_map_action_suppressed": int((_dsm_live.action.to_numpy() != state_action(_dsm_live.pred_state.to_numpy(int))).sum()),
    "bank_vs_live_state_changes": int((_dsm_bank.pred_state.to_numpy(int) != _dsm_live.pred_state.to_numpy(int)).sum()),
    "bank_vs_live_action_changes": int((_dsm_bank.action.to_numpy() != _dsm_live.action.to_numpy()).sum()),
    "budget_vs_state_map_live_state_changes": int((_dsm_budget_live.pred_state.to_numpy(int) != _dsm_live.pred_state.to_numpy(int)).sum()),
}])
assert DSM_LIVE_AUDIT.state_map_action_suppressed.eq(0).all()

for df, name in [(DSM_QA_SUMMARY, "QA_COMPARISON"), (DSM_QA_FOLDS, "QA_FOLD_COMPARISON"),
                 (DSM_PAIRED, "PAIRED_VS_BUDGET_E14_PROTOTYPE"),
                 (DSM_LIVE_ROUTING, "LIVE_ROUTING_METRICS"), (DSM_LIVE_AUDIT, "LIVE_ROUTING_AUDIT")]:
    write_csv(df, DSM_ROOT / (name + ".csv"))
display(DSM_QA_SUMMARY[["method", "n", "steered_EM", "Recovery", "Damage", "NetGain",
                         "intervention_rate", "retain_count", "accept_count"]])
display(DSM_PAIRED[["method", "reference", "DeltaEM", "CI_low", "CI_high"]])
display(DSM_LIVE_ROUTING)
display(DSM_LIVE_AUDIT)

_notes = [
    "# E17 추가실험 — Direction 판정 후 전부 개입", "",
    "## [해결책]", "Direction이 K+C-로 예측하면 retain, K-C+로 예측하면 accept, 나머지는 noop.",
    "추가 budget/확신도 필터를 없앴으며 head/readout/강도/레이어/split은 기존과 동일하다.",
    "전체 입력에 무조건 개입한 것이 아니라, 예측된 retain/accept 후보 모두에 개입했다.", "",
    "## 실제 QA", DSM_QA_SUMMARY.to_markdown(index=False), "",
    "## Paired 비교", DSM_PAIRED.to_markdown(index=False), "",
    "## 실제 생성 중 routing 진단", DSM_LIVE_ROUTING.to_markdown(index=False), "",
    "## 적용/일치 검사", DSM_LIVE_AUDIT.to_markdown(index=False), "",
    "## 해석", "주 비교는 direction__state_map vs direction__budget이다. 개입률·Recovery·Damage를 함께 본다.",
    "4-state F1은 동일 readout의 판별력이며 추가 개입률 제한 해제로 재학습되는 값이 아니다.",
    "state-derived action target은 QA 최적 action oracle이 아니다.",
    "이 조건은 기존 E17 결과를 본 뒤 사용자가 요청한 탐색적 후속 비교이며 새 blind test가 아니다.",
    "질문 단위 bootstrap CI는 고정된 예측에 조건부이며 전체 재학습/선택의 불확실성을 포함하지 않는다.",
    "원래 8개 조건의 cache와 보고서는 보존했다."]
atomic_text(DSM_ROOT / "DIRECTION_STATE_MAP_REPORT.md", "\n\n".join(_notes))
print("비교 보고서:", DSM_ROOT / "DIRECTION_STATE_MAP_REPORT.md")
